# 02 · EDA와 중복 그룹 보존 분할

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** Manifest의 유형·수위·방해물을 집계하고 이미지 SHA 그룹을 보존한 8:1:1 분할과 bbox 표본 검사를 만듭니다.
- **입력:** 누락된 01R 복구·최종화 단계의 승인된 v2 seal과 Manifest, 원천 ZIP
- **출력:** EDA 집계, train/val/test 분할, split/mission seal, 원 실행의 bbox 그림
- **실제 저장된 실행 상태:** 저장 출력은 01 Seal Gate PASS와 최종 PASS_WITH_QUARANTINE를 기록합니다. train=246,468 / val=30,808 / test=30,809이며 유형별 수량 검사가 통과했습니다. 입력인 01R 노트북은 복구되지 않아 공개본만으로 이 연결을 재현할 수 없습니다. 원본의 데이터 사진·그림 payload 8개는 공개본에서 제거했습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/02_eda_and_split.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/02_eda_and_split.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 고등부 예선 - Mission 1 EDA·24장 시각화·8:1:1 분할

이 노트북은 `DCC_Colab_01R_C0510_형식불일치_복구_최종화.ipynb`의 v2 최종 Seal을
입력 Gate로 사용해 Mission 1-1~1-5의 표, 그림, 분할 자산을 생성합니다.

## 실행 전 조건

- 01R의 마지막 출력이 `PASS_WITH_QUARANTINE_AND_APPROVED_WAIVER`여야 합니다.
- `raw_rows`와 `unique_stems`가 모두 308,085여야 합니다.
- 01R이 실행 중일 때 이 파일을 열어 둘 수 있으나, v2 최종 Seal이 생긴 뒤 실행하십시오.
- 첫 입력 Gate가 미완료·변조되었거나 승인 범위를 벗어나면 즉시 중단합니다.

## 이 파일이 수행하는 작업

1. Training 원천·라벨 ZIP 22쌍과 308,085 stem pair를 확인합니다.
2. 전수 Manifest에서 8개 목자판 유형과 15개 방해물 통계를 다시 계산합니다.
3. 최종 train split에서 A/B/C/E/F/G/H/S 각 3장씩 선택해 모든 유효 `mokja` bbox를 표시합니다.
4. 동일 이미지 SHA-256 그룹을 보존하고 각 유형 내부 정수 목표까지 고정해
   246,468/30,808/30,809로 분할합니다.
5. test inference와 test truth를 분리하고 split manifest를 포함한 모든 결과를 비공개
   GCS에 create-only로 봉인합니다.

공식 AI-Hub Validation ZIP은 이 노트북의 집계·분할·학습 자산에 포함하지 않습니다.
원본 ZIP은 읽기 전용이며, 전체 117.74 GiB를 loose file로 풀지 않습니다.


In [ ]:
# 사용자가 확인할 설정입니다. 같은 v3를 재실행할 때 값들을 유지하십시오.
PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"

INPUT_VERSION = "v2"
OUTPUT_VERSION = "v3"
RUN_ID = "20260825-mission1-v3"
SPLIT_SEED = 42

TARGET_COUNTS = {"train": 246_468, "val": 30_808, "test": 30_809}
TYPE_ORDER = list("ABCEFGHS")
WATER_BINS = 10
SAMPLES_PER_TYPE = 3

RESET_LOCAL_OUTPUT = True
CLEAN_IMAGE_STAGE_AFTER_FIGURES = True
RANGE_BLOCK_MIB = 4
RANGE_CACHE_BLOCKS = 8


## 1. 환경·인증

아래 셀은 01과 같은 핵심 I/O 패키지 버전을 확인하고 Google 계정 인증을 수행합니다.
패키지 교체 안내가 나오면 Colab 메뉴에서 런타임을 한 번 다시 시작한 뒤 첫 셀부터
실행하십시오. CPU 런타임으로 충분합니다.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as dist_version

if not (3, 10) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError(
        f"지원 Python은 3.10~3.14입니다: {sys.version_info.major}.{sys.version_info.minor}"
    )

REQUIRED_PACKAGES = {
    "google.cloud.storage": ("google-cloud-storage", "3.13.1"),
    "google_crc32c": ("google-crc32c", "1.8.0"),
    "pyarrow": ("pyarrow", "23.0.1"),
    "PIL": ("Pillow", "12.3.0"),
}

def module_available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ModuleNotFoundError):
        return False

def requirement_satisfied(distribution, expected_version):
    try:
        return dist_version(distribution) == expected_version
    except PackageNotFoundError:
        return False

needs_install = [
    (module, f"{distribution}=={expected_version}")
    for module, (distribution, expected_version) in REQUIRED_PACKAGES.items()
    if not module_available(module)
    or not requirement_satisfied(distribution, expected_version)
]
preloaded_needing_replacement = [
    module for module, _ in needs_install if module in sys.modules
]
if needs_install:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"]
        + [requirement for _, requirement in needs_install],
        check=True,
    )
    importlib.invalidate_caches()
    if preloaded_needing_replacement:
        raise RuntimeError(
            "교체한 패키지의 구버전 모듈이 메모리에 남아 있습니다: "
            f"{preloaded_needing_replacement}. Colab의 런타임 다시 시작을 누른 뒤 "
            "첫 셀부터 다시 실행하세요."
        )

unresolved = [
    f"{distribution}=={expected_version}"
    for distribution, expected_version in REQUIRED_PACKAGES.values()
    if not requirement_satisfied(distribution, expected_version)
]
if unresolved:
    raise RuntimeError(f"패키지 버전 요구사항을 충족하지 못했습니다: {unresolved}")

import base64
import csv
import gc
import hashlib
import io
import json
import math
import os
import platform
import re
import shutil
import tempfile
import warnings
import zipfile
from collections import Counter, OrderedDict, defaultdict
from importlib.metadata import version as package_version
from pathlib import Path

import google_crc32c
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display
from matplotlib.patches import Rectangle
from PIL import Image, ImageFile

from google.colab import auth

ImageFile.LOAD_TRUNCATED_IMAGES = False
Image.MAX_IMAGE_PIXELS = 1920 * 1080
IMPORTED_PACKAGE_VERSIONS = {
    "google-cloud-storage": storage.__version__,
    "google-crc32c": getattr(
        google_crc32c, "__version__", package_version("google-crc32c")
    ),
    "pyarrow": pa.__version__,
    "Pillow": PIL.__version__,
}
expected_imported_versions = {
    distribution: expected_version
    for distribution, expected_version in REQUIRED_PACKAGES.values()
}
if IMPORTED_PACKAGE_VERSIONS != expected_imported_versions:
    raise RuntimeError(
        "import된 패키지 버전이 lock과 다릅니다. 런타임을 다시 시작하세요: "
        f"actual={IMPORTED_PACKAGE_VERSIONS}, expected={expected_imported_versions}"
    )

if BUCKET_NAME.startswith("gs://"):
    raise ValueError("BUCKET_NAME에는 gs://를 넣지 마세요.")
if not re.fullmatch(r"v[0-9]+", INPUT_VERSION):
    raise ValueError("INPUT_VERSION은 v1 같은 형식이어야 합니다.")
if not re.fullmatch(r"v[0-9]+", OUTPUT_VERSION):
    raise ValueError("OUTPUT_VERSION은 v1 같은 형식이어야 합니다.")
if INPUT_VERSION != "v2" or OUTPUT_VERSION != "v3":
    raise ValueError("이 노트북의 승인 경계는 INPUT_VERSION=v2, OUTPUT_VERSION=v3입니다.")
if INPUT_VERSION == OUTPUT_VERSION:
    raise ValueError("입력과 출력 version은 달라야 합니다.")

auth.authenticate_user(project_id=PROJECT_ID)
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()
if str(BUCKET.location or "").upper() != "US-CENTRAL1":
    raise RuntimeError(f"예상하지 않은 bucket location: {BUCKET.location}")
if str(BUCKET.storage_class or "").upper() != "STANDARD":
    raise RuntimeError(f"예상하지 않은 storage class: {BUCKET.storage_class}")
if not BUCKET.iam_configuration.uniform_bucket_level_access_enabled:
    raise RuntimeError("Uniform bucket-level access가 활성화되어 있지 않습니다.")
public_access_prevention = getattr(
    BUCKET.iam_configuration, "public_access_prevention", None
)
if str(public_access_prevention or "").lower() != "enforced":
    raise RuntimeError(
        "Public access prevention이 enforced가 아닙니다: "
        f"{public_access_prevention}"
    )

RUNTIME_ENVIRONMENT = {
    "python": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
    **IMPORTED_PACKAGE_VERSIONS,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
}
print("Runtime:", json.dumps(RUNTIME_ENVIRONMENT, ensure_ascii=False, indent=2))
print("Platform:", platform.platform())
print("Bucket:", f"gs://{BUCKET_NAME}")
print(
    "Bucket policy:",
    BUCKET.location,
    BUCKET.storage_class,
    "uniform-access",
    public_access_prevention,
)


Runtime: {
  "python": "3.13.15",
  "google-cloud-storage": "3.13.1",
  "google-crc32c": "1.8.0",
  "pyarrow": "23.0.1",
  "Pillow": "12.3.0",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "matplotlib": "3.10.0"
}
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Bucket: gs://YOUR_GCS_BUCKET
Bucket policy: US-CENTRAL1 STANDARD uniform-access enforced


In [ ]:
NOTEBOOK_NAME = "DCC_Colab_02_Mission1_EDA_시각화_분할.ipynb"
PIPELINE_VERSION = "dcc-mission1-eda-split-1.1.0"
PIPELINE_BUILD_SHA256 = "6da311c19068d358306b12fd24afd8ba51cbfc7c358c54753dc0a0e01c46920e"
PARENT_PIPELINE_VERSION = "dcc-manifest-policy-repair-1.0.0"
PARENT_BUILD_SHA256 = "a8bc879b78858f59a85f3b1b1aae0c0cfc5e9c44cb7bc5fa279f939c4d05f0cb"
PARENT_SCHEMA_VERSION = "dcc-manifest-schema-v1"
PARENT_RUN_ID = "20260825-manifest-repair-v2"
PARENT_SUCCESS_STATUS = "PASS_WITH_QUARANTINE_AND_APPROVED_WAIVER"
WAIVER_KEY = "C_0510"
WAIVER_COUNT = 153
WAIVER_POLICY_ID = "quarantined-format-mismatch-waiver-v1"
PARENT_SEAL_OBJECT = f"dcc/audits/manifest_seal_{INPUT_VERSION}.json"

EXPECTED_RAW_ROWS = 308_085
EXPECTED_SOURCE_BYTES = 126_148_468_213
EXPECTED_LABEL_BYTES = 273_886_103
EXPECTED_SUFFIX_COUNTS = {".jpg": 298_587, ".jpeg": 9_498}
KNOWN_STRICT_DECODE_FAILURE_STEM = (
    "H_0007_6aa9acd6-6fad-4e74-8ad7-a6b2ae29cefe"
)
EXPECTED_ARCHIVE_COUNTS = {
    "A_0005": 24_068, "A_0510": 49_848, "A_1015": 31_304,
    "B_0003": 16_653, "B_0306": 13_811, "B_0609": 6_802,
    "C_0005": 22_335, "C_0510": 37_466, "C_1015": 32_140,
    "E_0003": 9_324, "E_0306": 7_008, "E_0610": 7_174,
    "F_0002": 3_404, "F_0204": 5_383,
    "G_0002": 4_666, "G_0204": 6_311,
    "H_0007": 2_839, "H_0714": 4_208, "H_1420": 3_264,
    "S_0007": 4_675, "S_0714": 6_378, "S_1420": 9_024,
}
EXPECTED_TYPE_COUNTS = {
    "A": 105_220, "B": 37_266, "C": 91_941, "E": 23_506,
    "F": 8_787, "G": 10_977, "H": 10_311, "S": 20_077,
}
TYPE_TARGET_POLICY = "hamilton-train-then-val-type-order-v1"
RAW_TYPE_SPLIT_TARGETS = {
    "A": {"train": 84_176, "val": 10_522, "test": 10_522},
    "B": {"train": 29_813, "val": 3_726, "test": 3_727},
    "C": {"train": 73_553, "val": 9_194, "test": 9_194},
    "E": {"train": 18_805, "val": 2_350, "test": 2_351},
    "F": {"train": 7_030, "val": 879, "test": 878},
    "G": {"train": 8_781, "val": 1_098, "test": 1_098},
    "H": {"train": 8_249, "val": 1_031, "test": 1_031},
    "S": {"train": 16_061, "val": 2_008, "test": 2_008},
}

def hamilton_allocate(population, seats, order):
    total = sum(int(population[name]) for name in order)
    if total <= 0 or seats < 0 or seats > total:
        raise ValueError("Hamilton 배분 입력이 잘못되었습니다.")
    allocated = {
        name: (int(population[name]) * int(seats)) // total
        for name in order
    }
    remaining = int(seats) - sum(allocated.values())
    ranked = sorted(
        order,
        key=lambda name: (
            -((int(population[name]) * int(seats)) % total),
            order.index(name),
        ),
    )
    for name in ranked[:remaining]:
        allocated[name] += 1
    return allocated

derived_train_targets = hamilton_allocate(
    EXPECTED_TYPE_COUNTS, TARGET_COUNTS["train"], TYPE_ORDER
)
remaining_after_train = {
    name: EXPECTED_TYPE_COUNTS[name] - derived_train_targets[name]
    for name in TYPE_ORDER
}
derived_val_targets = hamilton_allocate(
    remaining_after_train, TARGET_COUNTS["val"], TYPE_ORDER
)
derived_type_split_targets = {
    name: {
        "train": derived_train_targets[name],
        "val": derived_val_targets[name],
        "test": remaining_after_train[name] - derived_val_targets[name],
    }
    for name in TYPE_ORDER
}
assert derived_type_split_targets == RAW_TYPE_SPLIT_TARGETS
assert set(RAW_TYPE_SPLIT_TARGETS) == set(TYPE_ORDER)
assert all(
    sum(RAW_TYPE_SPLIT_TARGETS[name].values()) == EXPECTED_TYPE_COUNTS[name]
    for name in TYPE_ORDER
)
assert all(
    sum(RAW_TYPE_SPLIT_TARGETS[name][split] for name in TYPE_ORDER)
    == TARGET_COUNTS[split]
    for split in TARGET_COUNTS
)
assert all(
    abs(RAW_TYPE_SPLIT_TARGETS[name]["val"]
        - RAW_TYPE_SPLIT_TARGETS[name]["test"]) <= 1
    for name in TYPE_ORDER
)
EXPECTED_OBSTRUCTION_COUNTS = {
    "거센물살": 74_796, "거울반사": 70_704, "기타": 414,
    "달빛반사": 7_497, "반투명": 42_285, "부유물": 5_076,
    "부착물": 7_682, "사람": 1_676, "수풀": 24_099,
    "안개": 5_040, "탁류": 38_724, "표면손상": 7_944,
    "표면오염": 11_560, "야간": 9_657, "와류": 3_154,
    "해당없음": 97_134, "": 10,
}
OBSTRUCTION_ENGLISH = {
    "거센물살": "strong-current", "거울반사": "mirror-reflection",
    "기타": "other", "달빛반사": "moon-reflection",
    "반투명": "translucent", "부유물": "floating-debris",
    "부착물": "attached-object", "사람": "person", "수풀": "vegetation",
    "안개": "fog", "탁류": "turbid-water", "표면손상": "surface-damage",
    "표면오염": "surface-contamination", "야간": "night", "와류": "vortex",
    "해당없음": "none", "": "empty-tag",
}
EXPECTED_OBSTRUCTION_TAGS = 407_452
EXPECTED_MULTI_OBSTRUCTION_ROWS = 76_538
EXPECTED_OBSTRUCTION_CONTRADICTIONS = 574
EXPECTED_DUPLICATE_GROUPS = 2_364
EXPECTED_DUPLICATE_ROWS = 4_738
EXPECTED_TARGET_CONFLICT_GROUPS = 233

LOCAL_ROOT = Path("/content/dcc_mission1") / OUTPUT_VERSION
LOCAL_INPUT = Path("/content/dcc_mission1") / INPUT_VERSION / "input"
LOCAL_EDA = LOCAL_ROOT / "eda"
LOCAL_FIGURES = LOCAL_EDA / "figures"
LOCAL_TABLES = LOCAL_EDA / "tables"
LOCAL_PRIVATE = LOCAL_ROOT / "eval_private" / "manifests"
LOCAL_INFERENCE = LOCAL_ROOT / "inference" / "manifests"
LOCAL_STAGE = LOCAL_ROOT / "selected_images"
LOCAL_OUTPUT_RESET_ALLOWLIST = frozenset(
    Path(os.path.abspath(os.fspath(path)))
    for path in [
        LOCAL_EDA, LOCAL_PRIVATE.parent, LOCAL_INFERENCE.parent, LOCAL_STAGE,
    ]
)

def safe_reset_local_output(path):
    candidate = Path(os.path.abspath(os.fspath(path)))
    if candidate not in LOCAL_OUTPUT_RESET_ALLOWLIST:
        raise PermissionError(
            f"허용되지 않은 local output reset 경로: {candidate}"
        )
    if candidate.exists() or candidate.is_symlink():
        if candidate.is_symlink() or candidate.resolve(strict=True) != candidate:
            raise PermissionError(
                f"symlink가 포함된 local output reset 경로: {candidate}"
            )
        shutil.rmtree(candidate)

if RESET_LOCAL_OUTPUT:
    for output_root in [
        LOCAL_EDA, LOCAL_PRIVATE.parent, LOCAL_INFERENCE.parent, LOCAL_STAGE,
    ]:
        safe_reset_local_output(output_root)
for path in [
    LOCAL_INPUT, LOCAL_FIGURES, LOCAL_TABLES,
    LOCAL_PRIVATE, LOCAL_INFERENCE,
]:
    path.mkdir(parents=True, exist_ok=True)

def json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"JSON으로 변환할 수 없는 형식: {type(value)}")

def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=json_default,
    ).encode("utf-8")

def write_json_file(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            default=json_default,
        ),
        encoding="utf-8",
    )

def gs_uri(object_name):
    return f"gs://{BUCKET_NAME}/{object_name}"

def parse_gs_uri(uri):
    prefix = f"gs://{BUCKET_NAME}/"
    if not str(uri).startswith(prefix):
        raise ValueError(f"현재 bucket 밖의 URI입니다: {uri}")
    object_name = str(uri)[len(prefix):]
    if not object_name or object_name.startswith("/"):
        raise ValueError(f"잘못된 GCS URI입니다: {uri}")
    return object_name

def bytes_hashes(raw):
    crc = google_crc32c.Checksum()
    crc.update(raw)
    return {
        "size": len(raw),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "crc32c": base64.b64encode(crc.digest()).decode("ascii"),
    }

def local_file_hashes(path, chunk_size=16 * 1024**2):
    path = Path(path)
    sha = hashlib.sha256()
    crc = google_crc32c.Checksum()
    with path.open("rb") as reader:
        for block in iter(lambda: reader.read(chunk_size), b""):
            sha.update(block)
            crc.update(block)
    return {
        "size": path.stat().st_size,
        "sha256": sha.hexdigest(),
        "crc32c": base64.b64encode(crc.digest()).decode("ascii"),
    }

def verify_parent_artifact_descriptor(descriptor):
    required = {
        "object", "uri", "generation", "size", "crc32c", "sha256",
        "crc32c_local", "run_id", "config_sha256", "artifact_type",
    }
    missing = sorted(required - set(descriptor))
    if missing:
        raise IOError(f"parent artifact descriptor 필드 누락: {missing}")
    if descriptor["uri"] != gs_uri(descriptor["object"]):
        raise IOError(f"parent artifact URI 불일치: {descriptor['object']}")
    blob = BUCKET.blob(
        descriptor["object"], generation=int(descriptor["generation"])
    )
    blob.reload()
    metadata = blob.metadata or {}
    checks = {
        "generation": int(blob.generation) == int(descriptor["generation"]),
        "size": int(blob.size) == int(descriptor["size"]),
        "crc32c": (blob.crc32c or "") == descriptor["crc32c"],
        "crc_lineage": descriptor["crc32c_local"] == descriptor["crc32c"],
        "sha256_metadata": metadata.get("sha256") == descriptor["sha256"],
        "crc32c_metadata": metadata.get("crc32c_local") == descriptor["crc32c_local"],
        "run_id_metadata": metadata.get("run_id") == descriptor["run_id"],
        "config_metadata": metadata.get("config_sha256") == descriptor["config_sha256"],
        "artifact_metadata": metadata.get("artifact_type") == descriptor["artifact_type"],
    }
    if descriptor.get("row_count") is not None:
        checks["row_count_metadata"] = (
            metadata.get("row_count") == str(int(descriptor["row_count"]))
        )
    if not all(checks.values()):
        raise IOError(f"parent artifact 원격 검증 실패: {descriptor['uri']} / {checks}")
    return True

def download_parent_artifact(descriptor, destination):
    verify_parent_artifact_descriptor(descriptor)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        existing_hashes = local_file_hashes(destination)
        existing_checks = {
            "size": existing_hashes["size"] == int(descriptor["size"]),
            "sha256": existing_hashes["sha256"] == descriptor["sha256"],
            "crc32c": existing_hashes["crc32c"] == descriptor["crc32c"],
        }
        if all(existing_checks.values()):
            print("Reusing verified input:", descriptor["uri"])
            return destination
    part = destination.with_name(destination.name + ".part")
    if part.exists():
        part.unlink()
    blob = BUCKET.blob(
        descriptor["object"], generation=int(descriptor["generation"])
    )
    blob.download_to_filename(
        str(part),
        if_generation_match=int(descriptor["generation"]),
        checksum="crc32c",
    )
    hashes = local_file_hashes(part)
    checks = {
        "size": hashes["size"] == int(descriptor["size"]),
        "sha256": hashes["sha256"] == descriptor["sha256"],
        "crc32c": hashes["crc32c"] == descriptor["crc32c"],
    }
    if not all(checks.values()):
        part.unlink(missing_ok=True)
        raise IOError(f"parent artifact 다운로드 검증 실패: {descriptor['uri']} / {checks}")
    os.replace(part, destination)
    return destination


In [ ]:
# 01R의 v2 최종 Seal 한 개를 먼저 확인합니다. Seal이 없으면 부분 산출물은 사용하지 않습니다.
seal_blob = BUCKET.blob(PARENT_SEAL_OBJECT)
try:
    seal_blob.reload()
except NotFound:
    commit_prefix = "dcc/runs/20260825-manifest-v1/commits/"
    completed = sum(
        1 for blob in STORAGE_CLIENT.list_blobs(BUCKET, prefix=commit_prefix)
        if blob.name.endswith(".json")
    )
    raise RuntimeError(
        "01R 미완료: manifest_seal_v2.json이 아직 없습니다. "
        f"현재 확인 가능한 archive commit은 {completed}/22개입니다. "
        "01R의 마지막 셀이 승인된 v2 성공 상태로 끝난 뒤 재실행하세요."
    )

PARENT_SEAL_GENERATION = int(seal_blob.generation)
parent_raw = seal_blob.download_as_bytes(
    if_generation_match=PARENT_SEAL_GENERATION,
    checksum="crc32c",
)
parent_hashes = bytes_hashes(parent_raw)
parent_metadata = seal_blob.metadata or {}
PARENT_SEAL = json.loads(parent_raw.decode("utf-8"))
parent_seal_checks = {
    "size": int(seal_blob.size) == parent_hashes["size"],
    "gcs_crc32c": (seal_blob.crc32c or "") == parent_hashes["crc32c"],
    "sha256_metadata": parent_metadata.get("sha256") == parent_hashes["sha256"],
    "crc32c_metadata": parent_metadata.get("crc32c_local") == parent_hashes["crc32c"],
    "run_id_metadata": parent_metadata.get("run_id") == PARENT_SEAL.get("run_id"),
    "config_metadata": parent_metadata.get("config_sha256") == PARENT_SEAL.get("config_sha256"),
    "artifact_metadata": parent_metadata.get("artifact_type") == "manifest_seal",
    "status": PARENT_SEAL.get("status") == PARENT_SUCCESS_STATUS,
    "run_id": PARENT_SEAL.get("run_id") == PARENT_RUN_ID,
    "raw_content_download_bytes": PARENT_SEAL.get("raw_content_download_bytes") == 0,
    "observed_gate_failures": (
        PARENT_SEAL.get("observed_archive_gate_failures") == [WAIVER_KEY]
    ),
    "effective_gate_failures": (
        PARENT_SEAL.get("effective_archive_gate_failures") == []
    ),
    "schema": PARENT_SEAL.get("schema_version") == PARENT_SCHEMA_VERSION,
    "pipeline": PARENT_SEAL.get("pipeline_version") == PARENT_PIPELINE_VERSION,
    "build": PARENT_SEAL.get("pipeline_build_sha256") == PARENT_BUILD_SHA256,
    "output_version": PARENT_SEAL.get("output_version") == INPUT_VERSION,
    "project": PARENT_SEAL.get("project_id") == PROJECT_ID,
    "bucket": PARENT_SEAL.get("bucket_name") == BUCKET_NAME,
    "completed_keys": PARENT_SEAL.get("completed_keys") == list(EXPECTED_ARCHIVE_COUNTS),
    "inputs_unchanged": PARENT_SEAL.get("original_inputs_modified") is False,
}
validation = PARENT_SEAL.get("validation", {})
validation_counts = validation.get("counts", {})
PARENT_STRICT_DECODE_FAILURE_STEMS = tuple(
    validation.get("strict_decode_failure_stems", [])
)
PARENT_QUARANTINE_STEMS = tuple(validation.get("quarantine_stems", []))
parent_seal_checks.update({
    "raw_rows": validation_counts.get("raw_rows") == EXPECTED_RAW_ROWS,
    "unique_stems": validation_counts.get("unique_stems") == EXPECTED_RAW_ROWS,
    "quarantine_present": int(validation_counts.get("quarantine_count", 0)) >= WAIVER_COUNT,
    "observed_failure_count": validation_counts.get("archive_gate_failures") == 1,
    "observed_failure_keys": (
        validation_counts.get("archive_gate_failure_keys") == [WAIVER_KEY]
    ),
    "effective_failure_count": (
        validation_counts.get("archive_gate_failures_effective") == 0
    ),
    "validation_observed_failures": (
        validation.get("observed_archive_gate_failures") == [WAIVER_KEY]
    ),
    "validation_effective_failures": (
        validation.get("effective_archive_gate_failures") == []
    ),
    "hard_assertions": bool(validation.get("hard_assertions"))
    and all(validation["hard_assertions"].values()),
    "regression_assertions": bool(validation.get("regression_assertions"))
    and all(validation["regression_assertions"].values()),
    "strict_decode_evidence_nonempty": bool(PARENT_STRICT_DECODE_FAILURE_STEMS),
    "strict_decode_evidence_unique": (
        len(PARENT_STRICT_DECODE_FAILURE_STEMS)
        == len(set(PARENT_STRICT_DECODE_FAILURE_STEMS))
    ),
    "known_strict_decode_failure_present": (
        KNOWN_STRICT_DECODE_FAILURE_STEM in PARENT_STRICT_DECODE_FAILURE_STEMS
    ),
    "strict_decode_count_matches_evidence": (
        validation_counts.get("strict_decode_failures")
        == len(PARENT_STRICT_DECODE_FAILURE_STEMS)
    ),
    "strict_decode_quarantine_subset": (
        set(PARENT_STRICT_DECODE_FAILURE_STEMS) <= set(PARENT_QUARANTINE_STEMS)
    ),
})

policy_waivers = PARENT_SEAL.get("policy_waivers", [])
POLICY_WAIVER = policy_waivers[0] if len(policy_waivers) == 1 else {}
expected_gate_counts = {
    "json_parse_failure_count": 0,
    "json_schema_failure_count": 0,
    "strict_decode_failure_count": 0,
    "unexpected_resolution_count": 0,
    "image_format_failure_count": WAIVER_COUNT,
    "bbox_parse_error_count": 0,
}
expected_reason_counts = {
    "image_format_mismatch": WAIVER_COUNT,
    "jpeg_eoi_missing": WAIVER_COUNT,
    "jpeg_soi_missing": WAIVER_COUNT,
}
expected_evidence = {
    "strict_decode_ok": WAIVER_COUNT,
    "expected_resolution_ok": WAIVER_COUNT,
    "trainable_false": WAIVER_COUNT,
    "raw_inputs_modified": False,
}
expected_effective_gate = {
    "observed_archive_status": "AUDIT_COMPLETE_WITH_GATE_ERRORS",
    "observed_gate_pass": False,
    "policy_predicate_pass": True,
    "effective_gate_pass": True,
    "original_audit_preserved": True,
}
detected_format_counts = POLICY_WAIVER.get("detected_format_counts", {})
parent_run = PARENT_SEAL.get("parent_run", {})
parent_seal_checks.update({
    "one_policy_waiver": len(policy_waivers) == 1,
    "waiver_policy_id": POLICY_WAIVER.get("policy_id") == WAIVER_POLICY_ID,
    "waiver_archive": POLICY_WAIVER.get("archive_key") == WAIVER_KEY,
    "waiver_rows": POLICY_WAIVER.get("affected_rows") == WAIVER_COUNT,
    "waiver_disposition": (
        POLICY_WAIVER.get("disposition") == "preserve_raw_and_exclude_from_training"
    ),
    "waiver_raw_mutation": POLICY_WAIVER.get("raw_mutation_allowed") is False,
    "waiver_allowed_gate_counter": (
        POLICY_WAIVER.get("allowed_gate_counter")
        == {"image_format_failure_count": WAIVER_COUNT}
    ),
    "waiver_zero_counters": POLICY_WAIVER.get("required_zero_gate_counters") == [
        "json_parse_failure_count", "json_schema_failure_count",
        "strict_decode_failure_count", "unexpected_resolution_count",
        "bbox_parse_error_count",
    ],
    "waiver_required_reasons": POLICY_WAIVER.get("required_reason_codes") == [
        "image_format_mismatch", "jpeg_eoi_missing", "jpeg_soi_missing",
    ],
    "waiver_parent_run": POLICY_WAIVER.get("parent_run_id") == "20260825-manifest-v1",
    "waiver_parent_config": bool(re.fullmatch(
        r"[0-9a-f]{64}", str(POLICY_WAIVER.get("parent_config_sha256", ""))
    )),
    "waiver_parent_commit_object": (
        POLICY_WAIVER.get("parent_commit_object")
        == "dcc/runs/20260825-manifest-v1/commits/C_0510.json"
    ),
    "waiver_parent_commit_generation": (
        isinstance(POLICY_WAIVER.get("parent_commit_generation"), int)
        and POLICY_WAIVER["parent_commit_generation"] > 0
    ),
    "waiver_parent_commit_sha": bool(re.fullmatch(
        r"[0-9a-f]{64}", str(POLICY_WAIVER.get("parent_commit_sha256", ""))
    )),
    "waiver_gate_counts": POLICY_WAIVER.get("gate_counts") == expected_gate_counts,
    "waiver_reason_counts": POLICY_WAIVER.get("reason_counts") == expected_reason_counts,
    "waiver_evidence": POLICY_WAIVER.get("evidence") == expected_evidence,
    "waiver_effective_gate": (
        POLICY_WAIVER.get("effective_gate") == expected_effective_gate
    ),
    "waiver_stems_digest": bool(re.fullmatch(
        r"[0-9a-f]{64}", str(POLICY_WAIVER.get("affected_stems_sha256", ""))
    )),
    "waiver_detected_formats": (
        bool(detected_format_counts)
        and sum(int(value) for value in detected_format_counts.values()) == WAIVER_COUNT
        and all(str(name).upper() != "JPEG" for name in detected_format_counts)
    ),
    "parent_v1_run": parent_run.get("run_id") == "20260825-manifest-v1",
    "parent_run_config_object": (
        parent_run.get("run_config_object")
        == "dcc/runs/20260825-manifest-v1/run_config.json"
    ),
    "parent_run_config_generation": (
        isinstance(parent_run.get("run_config_generation"), int)
        and parent_run["run_config_generation"] > 0
    ),
    "parent_run_config_sha": bool(re.fullmatch(
        r"[0-9a-f]{64}", str(parent_run.get("run_config_sha256", ""))
    )),
    "parent_config_crosscheck": (
        parent_run.get("config_sha256") == POLICY_WAIVER.get("parent_config_sha256")
    ),
    "parent_commit_index": bool(re.fullmatch(
        r"[0-9a-f]{64}", str(parent_run.get("commit_index_sha256", ""))
    )),
})
if not all(parent_seal_checks.values()):
    failed = [name for name, passed in parent_seal_checks.items() if not passed]
    raise RuntimeError(f"01 최종 Seal Gate 실패: {failed}")

source_config_blob = BUCKET.blob(
    parent_run["run_config_object"],
    generation=int(parent_run["run_config_generation"]),
)
source_config_blob.reload()
source_config_raw = source_config_blob.download_as_bytes(
    if_generation_match=int(parent_run["run_config_generation"]),
    checksum="crc32c",
)
source_config_hashes = bytes_hashes(source_config_raw)
source_config_metadata = source_config_blob.metadata or {}
SOURCE_RUN_CONFIG = json.loads(source_config_raw.decode("utf-8"))
source_contract = SOURCE_RUN_CONFIG.get("pipeline_contract", {})
source_config_checks = {
    "generation": int(source_config_blob.generation)
    == int(parent_run["run_config_generation"]),
    "size": int(source_config_blob.size) == source_config_hashes["size"],
    "gcs_crc32c": (source_config_blob.crc32c or "")
    == source_config_hashes["crc32c"],
    "sha256": source_config_hashes["sha256"] == parent_run["run_config_sha256"],
    "sha256_metadata": source_config_metadata.get("sha256")
    == source_config_hashes["sha256"],
    "crc32c_metadata": source_config_metadata.get("crc32c_local")
    == source_config_hashes["crc32c"],
    "run_id_metadata": source_config_metadata.get("run_id")
    == "20260825-manifest-v1",
    "config_metadata": source_config_metadata.get("config_sha256")
    == parent_run["config_sha256"],
    "artifact_metadata": source_config_metadata.get("artifact_type") == "run_config",
    "payload_run_id": SOURCE_RUN_CONFIG.get("run_id") == "20260825-manifest-v1",
    "payload_config": SOURCE_RUN_CONFIG.get("config_sha256")
    == parent_run["config_sha256"],
    "config_digest": hashlib.sha256(
        canonical_json_bytes(source_contract)
    ).hexdigest() == parent_run["config_sha256"],
    "pipeline": source_contract.get("pipeline_version")
    == "dcc-full-manifest-audit-1.1.0",
    "build": source_contract.get("pipeline_build_sha256")
    == "dd582fa05398cf2b62e3b6249082263e02d862d089656c68ef85f8f77a3950f0",
    "schema": source_contract.get("schema_version") == PARENT_SCHEMA_VERSION,
    "output_version": source_contract.get("output_version") == "v1",
    "source_inventory": SOURCE_RUN_CONFIG.get("source_inventory")
    == PARENT_SEAL.get("source_inventory"),
    "source_inventory_sha": SOURCE_RUN_CONFIG.get("source_inventory_sha256")
    == PARENT_SEAL.get("source_inventory_sha256"),
}
if not all(source_config_checks.values()):
    failed = [name for name, passed in source_config_checks.items() if not passed]
    raise IOError(f"v1 parent run_config 검증 실패: {failed}")

source_inventory_rows = PARENT_SEAL.get("source_inventory", [])
recomputed_inventory_sha256 = hashlib.sha256(
    canonical_json_bytes(source_inventory_rows)
).hexdigest()
if recomputed_inventory_sha256 != PARENT_SEAL.get("source_inventory_sha256"):
    raise IOError("v2 Seal source inventory SHA-256 재계산 불일치")
expected_inventory_identities = {
    (f"dcc/raw/source/TS_{key}.zip", key, "source")
    for key in EXPECTED_ARCHIVE_COUNTS
} | {
    (f"dcc/raw/labels/TL_{key}.zip", key, "label")
    for key in EXPECTED_ARCHIVE_COUNTS
}
actual_inventory_identities = {
    (item.get("object"), item.get("key"), item.get("kind"))
    for item in source_inventory_rows
}
if actual_inventory_identities != expected_inventory_identities:
    raise IOError("v2 Seal source inventory 객체·key·kind 집합 불일치")
if len(source_inventory_rows) != 44:
    raise IOError(f"parent raw inventory가 44개가 아닙니다: {len(source_inventory_rows)}")
inventory_kind_counts = Counter(item.get("kind") for item in source_inventory_rows)
if inventory_kind_counts != {"source": 22, "label": 22}:
    raise IOError(f"parent raw inventory 종류 수 불일치: {inventory_kind_counts}")
if sum(int(item["size"]) for item in source_inventory_rows if item["kind"] == "source") != EXPECTED_SOURCE_BYTES:
    raise IOError("parent raw source byte 합이 기준값과 다릅니다.")
if sum(int(item["size"]) for item in source_inventory_rows if item["kind"] == "label") != EXPECTED_LABEL_BYTES:
    raise IOError("parent raw label byte 합이 기준값과 다릅니다.")
for item in source_inventory_rows:
    latest_blob = BUCKET.blob(item["object"])
    latest_blob.reload()
    raw_checks = {
        "uri": item.get("uri") == gs_uri(item["object"]),
        "generation": int(latest_blob.generation) == int(item["generation"]),
        "size": int(latest_blob.size) == int(item["size"]),
        "crc32c": (latest_blob.crc32c or "") == item["crc32c"],
    }
    if not all(raw_checks.values()):
        raise IOError(
            f"01 Seal 이후 raw 객체 상태가 변경되었습니다: {item['uri']} / {raw_checks}"
        )

PARENT_SEAL_SHA256 = parent_hashes["sha256"]
PIPELINE_CONTRACT = {
    "pipeline_version": PIPELINE_VERSION,
    "pipeline_build_sha256": PIPELINE_BUILD_SHA256,
    "notebook_name": NOTEBOOK_NAME,
    "input_version": INPUT_VERSION,
    "output_version": OUTPUT_VERSION,
    "run_id": RUN_ID,
    "parent_seal": {
        "object": PARENT_SEAL_OBJECT,
        "generation": PARENT_SEAL_GENERATION,
        "sha256": PARENT_SEAL_SHA256,
        "raw_manifest_sha256": PARENT_SEAL["artifacts"]["raw_manifest"]["sha256"],
        "policy_waiver_sha256": hashlib.sha256(
            canonical_json_bytes(POLICY_WAIVER)
        ).hexdigest(),
        "policy_id": WAIVER_POLICY_ID,
        "observed_gate_failures": [WAIVER_KEY],
        "effective_gate_failures": [],
    },
    "runtime": RUNTIME_ENVIRONMENT,
    "seed": SPLIT_SEED,
    "targets": TARGET_COUNTS,
    "raw_type_split_targets": RAW_TYPE_SPLIT_TARGETS,
    "type_target_policy": TYPE_TARGET_POLICY,
    "types": TYPE_ORDER,
    "water_bins": WATER_BINS,
    "group_policy": "exact-image-sha256-v1",
    "stratification": "type-plus-stable-waterlevel-decile-v1",
    "fold_policy": "stable-balanced-10-fold-then-type-exact-singleton-rebalance-v2",
    "forced_train_policy": "group-containing-nontrainable-or-strict-failure-v1",
    "official_validation_allowed": False,
    "raw_write_allowed": False,
}
CONFIG_SHA256 = hashlib.sha256(canonical_json_bytes(PIPELINE_CONTRACT)).hexdigest()

parent_key_commits = PARENT_SEAL.get("key_commits", {})
if set(parent_key_commits) != set(EXPECTED_ARCHIVE_COUNTS):
    raise IOError("v2 Seal의 parent key commit 집합이 22개 기준과 다릅니다.")
for key, descriptor in parent_key_commits.items():
    commit_checks = {
        "object": descriptor.get("object")
        == f"dcc/runs/20260825-manifest-v1/commits/{key}.json",
        "run_id": descriptor.get("run_id") == "20260825-manifest-v1",
        "config": descriptor.get("config_sha256") == parent_run.get("config_sha256"),
        "type": descriptor.get("artifact_type") == "key_commit",
        "payload_sha": bool(re.fullmatch(
            r"[0-9a-f]{64}", str(descriptor.get("payload_sha256", ""))
        )),
    }
    if not all(commit_checks.values()):
        raise IOError(f"v2 Seal parent commit identity 불일치: {key} / {commit_checks}")
    verify_parent_artifact_descriptor(descriptor)

commit_index_payload = {
    key: {
        "object": parent_key_commits[key]["object"],
        "generation": parent_key_commits[key]["generation"],
        "sha256": parent_key_commits[key]["sha256"],
        "payload_sha256": parent_key_commits[key]["payload_sha256"],
    }
    for key in EXPECTED_ARCHIVE_COUNTS
}
recomputed_commit_index_sha256 = hashlib.sha256(
    canonical_json_bytes(commit_index_payload)
).hexdigest()
if recomputed_commit_index_sha256 != parent_run.get("commit_index_sha256"):
    raise IOError("v2 Seal의 parent commit index SHA-256 재계산 불일치")
c0510_commit_descriptor = parent_key_commits[WAIVER_KEY]
c0510_waiver_commit_checks = {
    "object": POLICY_WAIVER.get("parent_commit_object")
    == c0510_commit_descriptor.get("object"),
    "generation": int(POLICY_WAIVER.get("parent_commit_generation", -1))
    == int(c0510_commit_descriptor.get("generation", -2)),
    "sha256": POLICY_WAIVER.get("parent_commit_sha256")
    == c0510_commit_descriptor.get("sha256"),
    "config": POLICY_WAIVER.get("parent_config_sha256")
    == c0510_commit_descriptor.get("config_sha256"),
}
if not all(c0510_waiver_commit_checks.values()):
    raise IOError(
        f"C_0510 waiver와 parent commit descriptor 불일치: "
        f"{c0510_waiver_commit_checks}"
    )

parent_artifacts = PARENT_SEAL.get("artifacts", {})
required_parent_names = {
    "raw_manifest", "archive_audit", "bbox_quality", "quarantine",
    "format_mismatch_waiver",
}
mission_input_names = {
    "raw_manifest", "archive_audit", "bbox_quality", "quarantine"
}
expected_parent_artifact_objects = {
    "raw_manifest": f"dcc/manifests/raw_manifest_{INPUT_VERSION}.parquet",
    "archive_audit": f"dcc/audits/archive_audit_{INPUT_VERSION}.parquet",
    "image_integrity": f"dcc/audits/image_integrity_{INPUT_VERSION}.parquet",
    "bbox_quality": f"dcc/audits/bbox_quality_{INPUT_VERSION}.parquet",
    "quarantine": f"dcc/audits/quarantine_{INPUT_VERSION}.json",
    "archive_summary_csv": f"dcc/audits/archive_audit_summary_{INPUT_VERSION}.csv",
    "format_mismatch_waiver": f"dcc/audits/format_mismatch_waiver_{INPUT_VERSION}.json",
}
if set(parent_artifacts) != set(expected_parent_artifact_objects):
    raise IOError("01R Seal의 canonical artifact 집합이 v2 계약과 다릅니다.")
for artifact_name, expected_object in expected_parent_artifact_objects.items():
    descriptor = parent_artifacts[artifact_name]
    artifact_checks = {
        "object": descriptor.get("object") == expected_object,
        "run_id": descriptor.get("run_id") == PARENT_RUN_ID,
        "config": descriptor.get("config_sha256") == PARENT_SEAL.get("config_sha256"),
        "type": descriptor.get("artifact_type") == artifact_name,
    }
    if not all(artifact_checks.values()):
        raise IOError(
            f"01R artifact identity 불일치: {artifact_name} / {artifact_checks}"
        )
    verify_parent_artifact_descriptor(descriptor)

required_download_bytes = sum(
    int(parent_artifacts[name]["size"]) for name in mission_input_names
)
free_bytes = shutil.disk_usage("/content").free
if free_bytes < required_download_bytes + 8 * 1024**3:
    raise RuntimeError(
        "Mission 1 local 공간 부족: "
        f"free={free_bytes/1024**3:.1f} GiB, "
        f"input={required_download_bytes/1024**3:.1f} GiB, buffer=8.0 GiB"
    )

INPUT_PATHS = {
    "raw_manifest": LOCAL_INPUT / f"raw_manifest_{INPUT_VERSION}.parquet",
    "archive_audit": LOCAL_INPUT / f"archive_audit_{INPUT_VERSION}.parquet",
    "bbox_quality": LOCAL_INPUT / f"bbox_quality_{INPUT_VERSION}.parquet",
    "quarantine": LOCAL_INPUT / f"quarantine_{INPUT_VERSION}.json",
}
waiver_artifact_path = LOCAL_INPUT / f"format_mismatch_waiver_{INPUT_VERSION}.json"
download_parent_artifact(
    parent_artifacts["format_mismatch_waiver"], waiver_artifact_path
)
waiver_artifact_payload = json.loads(waiver_artifact_path.read_text(encoding="utf-8"))
if waiver_artifact_payload != POLICY_WAIVER:
    raise IOError("v2 waiver artifact와 Seal의 policy_waivers가 일치하지 않습니다.")

for name, path in INPUT_PATHS.items():
    print("Downloading verified input:", parent_artifacts[name]["uri"])
    download_parent_artifact(parent_artifacts[name], path)

manifest_file = pq.ParquetFile(INPUT_PATHS["raw_manifest"])
bbox_file = pq.ParquetFile(INPUT_PATHS["bbox_quality"])
audit_file = pq.ParquetFile(INPUT_PATHS["archive_audit"])
manifest_fields = set(manifest_file.schema_arrow.names)
required_manifest_fields = {
    "archive_key", "stem", "type", "clslevel", "waterlevel_m",
    "source_zip_uri", "source_generation", "source_member",
    "source_member_crc32", "source_member_bytes", "image_suffix",
    "width", "height", "image_sha256", "strict_decode_ok", "trainable",
    "detector_trainable", "obstruction", "mokja_count", "valid_mokja_count",
}
required_bbox_fields = {
    "stem", "annotation_name", "usable_after_clip", "out_of_range",
    "degenerate", "clipped_x1", "clipped_y1", "clipped_x2", "clipped_y2",
}
if not required_manifest_fields <= manifest_fields:
    raise TypeError(
        f"raw manifest 필드 누락: {sorted(required_manifest_fields - manifest_fields)}"
    )
if not required_bbox_fields <= set(bbox_file.schema_arrow.names):
    raise TypeError(
        f"bbox 필드 누락: {sorted(required_bbox_fields - set(bbox_file.schema_arrow.names))}"
    )
assert manifest_file.metadata.num_rows == EXPECTED_RAW_ROWS
assert audit_file.metadata.num_rows == 22
assert bbox_file.metadata.num_rows == 3_418_229

print("01 Seal Gate: PASS")
print("Parent status:", PARENT_SEAL["status"])
print("Parent Seal SHA-256:", PARENT_SEAL_SHA256)
print("Mission 1 config SHA-256:", CONFIG_SHA256)


Reusing verified input: gs://YOUR_GCS_BUCKET/dcc/audits/format_mismatch_waiver_v2.json
Reusing verified input: gs://YOUR_GCS_BUCKET/dcc/manifests/raw_manifest_v2.parquet
Reusing verified input: gs://YOUR_GCS_BUCKET/dcc/audits/archive_audit_v2.parquet
Reusing verified input: gs://YOUR_GCS_BUCKET/dcc/audits/bbox_quality_v2.parquet
Reusing verified input: gs://YOUR_GCS_BUCKET/dcc/audits/quarantine_v2.json
01 Seal Gate: PASS
Parent status: PASS_WITH_QUARANTINE_AND_APPROVED_WAIVER
Parent Seal SHA-256: f657b216d7fdc6c257d4544f209d19be0f1f597600cc6e7122693ab546d49e98
Mission 1 config SHA-256: 5e06e341e73fb42c52a0a424493f1f88943470e59f73dce338e06712103384a4


## 2. Mission 1-1·1-2 - 수량, 유형, 방해물 전수 집계

아래 집계는 01에서 모든 Training JSON과 이미지를 검사해 만든 raw Manifest를 다시 읽어
계산합니다. `obstruction`은 multi-label list이므로 태그 발생 합계와 이미지 수가 같을 필요가
없습니다. `해당없음`은 비방해 범주로, 빈 문자열 10개는 품질 이상으로 따로 기록합니다.

`H_0007`의 알려진 strict decode 실패 1개를 포함해 v2 Seal에 기록된 모든 strict
decode 실패는 격리 상태를 유지하고 집계에는 보존하되, 학습·검증·테스트의 유효
입력에서는 제외합니다.

E의 마지막 구간은 PDF 표기에 6-9m로 제시되어 있고 실제 ZIP 이름은 `E_0610`, JSON의
`clslevel`은 6~10m입니다. 표와 후속 계산은 실제 JSON 값을 사용합니다.


In [ ]:
audit = pq.read_table(INPUT_PATHS["archive_audit"]).to_pandas()
audit = audit.sort_values("archive_key", kind="mergesort").reset_index(drop=True)
archive_counts = audit[[
    "archive_key", "source_zip_uri", "label_zip_uri",
    "source_actual_bytes", "label_actual_bytes",
    "source_member_count", "label_member_count", "pair_count",
    "only_image_count", "only_json_count", "status", "gate_pass",
    "json_parse_failure_count", "json_schema_failure_count",
    "strict_decode_failure_count", "unexpected_resolution_count",
    "image_format_failure_count", "quarantine_count", "bbox_parse_error_count",
]].copy()
archive_counts["official_validation"] = (
    archive_counts["source_zip_uri"].str.contains(r"/VS_", regex=True)
    | archive_counts["label_zip_uri"].str.contains(r"/VL_", regex=True)
)

assert len(archive_counts) == 22
assert archive_counts["archive_key"].tolist() == list(EXPECTED_ARCHIVE_COUNTS)
assert archive_counts.set_index("archive_key")["pair_count"].to_dict() == EXPECTED_ARCHIVE_COUNTS
assert int(archive_counts["source_member_count"].sum()) == EXPECTED_RAW_ROWS
assert int(archive_counts["label_member_count"].sum()) == EXPECTED_RAW_ROWS
assert int(archive_counts["pair_count"].sum()) == EXPECTED_RAW_ROWS
assert int(archive_counts["source_actual_bytes"].sum()) == EXPECTED_SOURCE_BYTES
assert int(archive_counts["label_actual_bytes"].sum()) == EXPECTED_LABEL_BYTES
assert int(archive_counts["only_image_count"].sum()) == 0
assert int(archive_counts["only_json_count"].sum()) == 0
observed_gate_failure_keys = archive_counts.loc[
    ~archive_counts["gate_pass"], "archive_key"
].tolist()
assert observed_gate_failure_keys == [WAIVER_KEY]
archive_counts["observed_gate_pass"] = archive_counts["gate_pass"]
archive_counts["effective_gate_pass"] = (
    archive_counts["gate_pass"] | archive_counts["archive_key"].eq(WAIVER_KEY)
)
archive_counts["waiver_id"] = np.where(
    archive_counts["archive_key"].eq(WAIVER_KEY), WAIVER_POLICY_ID, ""
)
assert bool(archive_counts["effective_gate_pass"].all())
c0510_audit = archive_counts.set_index("archive_key").loc[WAIVER_KEY]
assert c0510_audit["status"] == "AUDIT_COMPLETE_WITH_GATE_ERRORS"
assert bool(c0510_audit["gate_pass"]) is False
assert int(c0510_audit["pair_count"]) == 37_466
assert int(c0510_audit["quarantine_count"]) == WAIVER_COUNT
assert int(c0510_audit["image_format_failure_count"]) == WAIVER_COUNT
for field in [
    "json_parse_failure_count", "json_schema_failure_count",
    "unexpected_resolution_count", "bbox_parse_error_count",
]:
    assert int(c0510_audit[field]) == 0
    assert int(archive_counts[field].sum()) == 0
assert int(c0510_audit["strict_decode_failure_count"]) == 0
assert (
    int(archive_counts["strict_decode_failure_count"].sum())
    == len(PARENT_STRICT_DECODE_FAILURE_STEMS)
)
assert int(archive_counts["image_format_failure_count"].sum()) == WAIVER_COUNT
assert int(archive_counts["official_validation"].sum()) == 0
assert archive_counts["source_zip_uri"].str.contains(
    r"/dcc/raw/source/TS_[A-Z]_[0-9]{4}\.zip$", regex=True
).all()
assert archive_counts["label_zip_uri"].str.contains(
    r"/dcc/raw/labels/TL_[A-Z]_[0-9]{4}\.zip$", regex=True
).all()

CORE_COLUMNS = [
    "archive_key", "stem", "type", "clslevel", "waterlevel_m",
    "source_zip_uri", "source_generation", "source_member",
    "source_member_crc32", "source_member_bytes", "image_suffix",
    "width", "height", "image_sha256", "strict_decode_ok", "trainable",
    "detector_trainable", "obstruction", "mokja_count", "valid_mokja_count",
]
core = pq.read_table(
    INPUT_PATHS["raw_manifest"], columns=CORE_COLUMNS
).to_pandas()
core = core.sort_values("stem", kind="mergesort").reset_index(drop=True)
assert len(core) == EXPECTED_RAW_ROWS
assert core["stem"].is_unique
assert set(core["type"]) == set(TYPE_ORDER)
assert core["image_sha256"].str.fullmatch(r"[0-9a-f]{64}").all()
assert core["waterlevel_m"].map(math.isfinite).all()
assert (core["stem"].str[0] == core["type"]).all()
strict_decode_failure_stems = set(
    core.loc[~core["strict_decode_ok"], "stem"]
)
assert strict_decode_failure_stems == set(PARENT_STRICT_DECODE_FAILURE_STEMS)
assert (
    int(archive_counts["strict_decode_failure_count"].sum())
    == len(strict_decode_failure_stems)
)

def normalized_obstructions(value):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, (list, tuple)):
        raise TypeError(f"obstruction list 형식 위반: {type(value)}")
    return [str(item) for item in value]

core["obstruction"] = core["obstruction"].map(normalized_obstructions)
suffix_counts = core["image_suffix"].value_counts().sort_index().to_dict()
archive_manifest_counts = core["archive_key"].value_counts().to_dict()
type_counts_dict = core["type"].value_counts().to_dict()
assert suffix_counts == EXPECTED_SUFFIX_COUNTS
assert archive_manifest_counts == EXPECTED_ARCHIVE_COUNTS
assert type_counts_dict == EXPECTED_TYPE_COUNTS

type_range_counts = (
    core.groupby(["type", "archive_key", "clslevel"], observed=True)
    .size().rename("count").reset_index()
    .sort_values(["type", "archive_key"], kind="mergesort")
    .reset_index(drop=True)
)
type_counts = (
    core.groupby("type", observed=True).size().rename("count")
    .reindex(TYPE_ORDER).reset_index()
)

obstruction_counter = Counter()
multi_obstruction_rows = 0
obstruction_contradictions = 0
for tags in core["obstruction"]:
    obstruction_counter.update(tags)
    multi_obstruction_rows += int(len(tags) > 1)
    real = [tag for tag in tags if tag not in {"", "해당없음"}]
    obstruction_contradictions += int("해당없음" in tags and bool(real))
assert dict(obstruction_counter) == EXPECTED_OBSTRUCTION_COUNTS
assert sum(obstruction_counter.values()) == EXPECTED_OBSTRUCTION_TAGS
assert multi_obstruction_rows == EXPECTED_MULTI_OBSTRUCTION_ROWS
assert obstruction_contradictions == EXPECTED_OBSTRUCTION_CONTRADICTIONS
real_obstruction_names = sorted(
    name for name in obstruction_counter if name not in {"", "해당없음"}
)
assert len(real_obstruction_names) == 15

obstruction_rows = []
for name in real_obstruction_names:
    obstruction_rows.append({
        "category": "obstruction", "raw_tag": name,
        "count": obstruction_counter[name],
    })
obstruction_rows.extend([
    {"category": "none", "raw_tag": "해당없음", "count": obstruction_counter["해당없음"]},
    {"category": "quality_issue", "raw_tag": "<empty-string>", "count": obstruction_counter[""]},
])
obstruction_counts = pd.DataFrame(obstruction_rows)

group_sizes = core.groupby("image_sha256", observed=True).size()
duplicate_groups = int(group_sizes.gt(1).sum())
duplicate_rows = int(group_sizes[group_sizes.gt(1)].sum())
target_conflict_groups = int(
    core.groupby("image_sha256", observed=True)["waterlevel_m"]
    .nunique().gt(1).sum()
)
assert duplicate_groups == EXPECTED_DUPLICATE_GROUPS
assert duplicate_rows == EXPECTED_DUPLICATE_ROWS
assert target_conflict_groups == EXPECTED_TARGET_CONFLICT_GROUPS

quality_summary = pd.DataFrame([
    {"metric": "training_images", "value": EXPECTED_RAW_ROWS},
    {"metric": "training_json", "value": EXPECTED_RAW_ROWS},
    {"metric": "paired_stems", "value": EXPECTED_RAW_ROWS},
    {"metric": "official_validation_rows", "value": 0},
    {"metric": "obstruction_tag_occurrences", "value": sum(obstruction_counter.values())},
    {"metric": "multi_obstruction_images", "value": multi_obstruction_rows},
    {"metric": "none_with_real_obstruction", "value": obstruction_contradictions},
    {"metric": "empty_obstruction_tags", "value": obstruction_counter[""]},
    {"metric": "exact_duplicate_groups", "value": duplicate_groups},
    {"metric": "exact_duplicate_rows", "value": duplicate_rows},
    {"metric": "target_conflict_groups", "value": target_conflict_groups},
])

archive_counts.to_csv(LOCAL_TABLES / f"archive_counts_{OUTPUT_VERSION}.csv", index=False, encoding="utf-8-sig")
type_range_counts.to_csv(LOCAL_TABLES / f"type_range_counts_{OUTPUT_VERSION}.csv", index=False, encoding="utf-8-sig")
type_counts.to_csv(LOCAL_TABLES / f"type_counts_{OUTPUT_VERSION}.csv", index=False, encoding="utf-8-sig")
obstruction_counts.to_csv(LOCAL_TABLES / f"obstruction_counts_{OUTPUT_VERSION}.csv", index=False, encoding="utf-8-sig")
quality_summary.to_csv(LOCAL_TABLES / f"quality_summary_{OUTPUT_VERSION}.csv", index=False, encoding="utf-8-sig")

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(type_counts["type"], type_counts["count"], color="#3973ac")
ax.set_title("Training images by gauge type")
ax.set_xlabel("Gauge type")
ax.set_ylabel("Images")
for x, value in enumerate(type_counts["count"]):
    ax.text(x, value, f"{int(value):,}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(
    LOCAL_FIGURES / f"type_counts_{OUTPUT_VERSION}.png",
    dpi=140, metadata={"Software": PIPELINE_VERSION},
)
plt.close(fig)

real_plot = obstruction_counts.query("category == 'obstruction'").copy()
real_plot = real_plot.sort_values("count", ascending=False, kind="mergesort").reset_index(drop=True)
real_plot["rank"] = np.arange(1, len(real_plot) + 1)
real_plot[["rank", "raw_tag", "count"]].to_csv(
    LOCAL_TABLES / f"obstruction_plot_key_{OUTPUT_VERSION}.csv",
    index=False, encoding="utf-8-sig",
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(real_plot["rank"].astype(str), real_plot["count"], color="#d17c2f")
ax.set_title("15 obstruction tags (rank mapping is saved as CSV)")
ax.set_xlabel("Rank")
ax.set_ylabel("Tag occurrences")
fig.tight_layout()
fig.savefig(
    LOCAL_FIGURES / f"obstruction_counts_{OUTPUT_VERSION}.png",
    dpi=140, metadata={"Software": PIPELINE_VERSION},
)
plt.close(fig)

display(archive_counts)
display(type_range_counts)
display(type_counts)
display(obstruction_counts.sort_values(["category", "count"], ascending=[True, False]))
display(quality_summary)
print("Mission 1-1/1-2 assertions: PASS")


   archive_key                                     source_zip_uri  \
0       A_0005  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
1       A_0510  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
2       A_1015  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
3       B_0003  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
4       B_0306  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
5       B_0609  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
6       C_0005  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
7       C_0510  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
8       C_1015  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
9       E_0003  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
10      E_0306  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
11      E_0610  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
12      F_0002  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
13      F_0204  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
14      G_0002  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
15      G_0204  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
16      H_0007  gs://YOUR_GCS_BUCKET/dcc/raw/s...   
17      H_0714  gs://YOUR_GCS_

   type archive_key clslevel  count
0     A      A_0005     0~5m  24068
1     A      A_0510    5~10m  49848
2     A      A_1015   10~15m  31304
3     B      B_0003     0~3m  16653
4     B      B_0306     3~6m  13811
5     B      B_0609     6~9m   6802
6     C      C_0005     0~5m  22335
7     C      C_0510    5~10m  37466
8     C      C_1015   10~15m  32140
9     E      E_0003     0~3m   9324
10    E      E_0306     3~6m   7008
11    E      E_0610    6~10m   7174
12    F      F_0002     0~2m   3404
13    F      F_0204     2~4m   5383
14    G      G_0002     0~2m   4666
15    G      G_0204     2~4m   6311
16    H      H_0007     0~7m   2839
17    H      H_0714    7~14m   4208
18    H      H_1420   14~20m   3264
19    S      S_0007     0~7m   4675
20    S      S_0714    7~14m   6378
21    S      S_1420   14~20m   9024

  type   count
0    A  105220
1    B   37266
2    C   91941
3    E   23506
4    F    8787
5    G   10977
6    H   10311
7    S   20077

         category         raw_tag  count
15           none            해당없음  97134
0     obstruction            거센물살  74796
1     obstruction            거울반사  70704
4     obstruction             반투명  42285
12    obstruction              탁류  38724
8     obstruction              수풀  24099
14    obstruction            표면오염  11560
10    obstruction              야간   9657
13    obstruction            표면손상   7944
6     obstruction             부착물   7682
3     obstruction            달빛반사   7497
5     obstruction             부유물   5076
9     obstruction              안개   5040
11    obstruction              와류   3154
7     obstruction              사람   1676
2     obstruction              기타    414
16  quality_issue  <empty-string>     10

                         metric   value
0               training_images  308085
1                 training_json  308085
2                  paired_stems  308085
3      official_validation_rows       0
4   obstruction_tag_occurrences  407452
5      multi_obstruction_images   76538
6    none_with_real_obstruction     574
7        empty_obstruction_tags      10
8        exact_duplicate_groups    2364
9          exact_duplicate_rows    4738
10       target_conflict_groups     233

Mission 1-1/1-2 assertions: PASS


## 3. Mission 1-4 - exact-duplicate group을 보존한 8:1:1 분할

PDF는 전체 Training의 8:1:1 비율을 요구하며 seed와 홀수 반올림 규칙은 지정하지
않습니다. 이 파일은 train 246,468, validation 30,808, test 30,809를 사용하고,
Hamilton 최대잔여법으로 A/B/C/E/F/G/H/S 각 유형의 정수 목표도 정확히 고정합니다.
train을 먼저 배분하고 남은 행에서 validation을 배분하며, 동률은 유형 순서로 정합니다.

`group_id`는 실제 이미지 SHA-256입니다. 유형과 유형 내 수위 decile을 최대한 유지하는
10개 fold를 안정 해시 순서로 만든 뒤, 같은 유형의 전역 singleton group만 이동해
유형×split 목표 수량을 정확히 맞춥니다.
중복 group은 이동 과정에서도 분해하지 않습니다. pHash 근접중복 clustering은 이 단계의
구현 범위에 포함하지 않으며, 해당 효과를 결과로 주장하지 않습니다.

실행 의존성 때문에 Mission 1-4 분할 셀이 Mission 1-3 그림 셀보다 먼저 배치됩니다.
이렇게 하면 24장 시각화 표본을 확정된 train split 안에서 선택할 수 있습니다.

validation은 모델·epoch·threshold 선택에 사용하고, test는 최종 평가 1회에 사용합니다.
`test_inference.image_path`는 파일을 여는 I/O 경로로만 쓰며 문자열 feature로 변환하지
않습니다. 허용된 유형 정보는 stem 첫 문자에서 만들고, Mission 4 단계에서는 type도
모델 입력에서 제외합니다.

이 셀은 분할 Parquet을 Colab 로컬에 생성합니다. GCS 영구 게시는 Mission 1-5의 마지막
게시 셀이 담당하며, `MISSION 1 COMPLETE: GCS_PUBLISH_PASS` 전에는 완료로 판단하지 않습니다.


In [ ]:
def stable_hex(seed, namespace, value):
    payload = f"{seed}|{namespace}|{value}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def add_stable_water_bins(frame, bins=10):
    out = frame.copy()
    out["water_bin"] = -1
    for type_name, indices in out.groupby("type", sort=True).groups.items():
        ordered = (
            out.loc[indices, ["waterlevel_m", "stem"]]
            .sort_values(["waterlevel_m", "stem"], kind="mergesort")
            .index.to_numpy()
        )
        n = len(ordered)
        assigned = np.minimum((np.arange(n, dtype=np.int64) * bins) // n, bins - 1)
        out.loc[ordered, "water_bin"] = assigned
    if out["water_bin"].lt(0).any():
        raise AssertionError("water_bin 미배정 행이 있습니다.")
    out["water_bin"] = out["water_bin"].astype("int8")
    out["stratum"] = (
        out["type"] + "_" + out["water_bin"].astype(str).str.zfill(2)
    )
    return out

def assign_balanced_group_folds(frame, n_folds=10, seed=42):
    group_sizes_local = (
        frame.groupby("group_id", observed=True).size().rename("group_size")
    )
    group_strata = (
        frame.groupby(["group_id", "stratum"], observed=True)
        .size().rename("rows_in_stratum").reset_index()
        .sort_values(
            ["group_id", "rows_in_stratum", "stratum"],
            ascending=[True, False, True], kind="mergesort",
        )
        .drop_duplicates("group_id")
        .set_index("group_id")["stratum"]
    )
    groups = pd.concat([group_sizes_local, group_strata], axis=1).reset_index()
    mapping = {}
    for stratum, part in groups.groupby("stratum", sort=True):
        records = [
            (int(row.group_size), str(row.group_id))
            for row in part.itertuples(index=False)
        ]
        records.sort(
            key=lambda item: (
                -item[0], stable_hex(seed, f"fold-order:{stratum}", item[1])
            )
        )
        loads = [0] * n_folds
        for group_size, group_id in records:
            minimum = min(loads)
            candidates = [fold for fold, load in enumerate(loads) if load == minimum]
            tie = int(stable_hex(seed, f"fold-tie:{stratum}", group_id)[:16], 16)
            fold = candidates[tie % len(candidates)]
            mapping[group_id] = fold
            loads[fold] += group_size
    if len(mapping) != frame["group_id"].nunique():
        raise AssertionError("group fold mapping이 완전하지 않습니다.")
    return mapping

def _legacy_global_only_rebalance_unused(
    frame, targets, split_col="split", group_col="group_id",
    stratum_col="stratum", seed=42, protected_groups=None,
):
    out = frame.copy()
    split_names = list(targets)
    protected_groups = set() if protected_groups is None else set(protected_groups)
    if len(out) != sum(targets.values()):
        raise ValueError("target 합과 전체 행 수가 다릅니다.")
    group_sizes_local = out.groupby(group_col, observed=True).size()
    singleton_mask = out[group_col].map(group_sizes_local).eq(1)
    singleton_mask &= ~out[group_col].isin(protected_groups)
    current = (
        out[split_col].value_counts().reindex(split_names, fill_value=0)
        .astype(int).to_dict()
    )
    strata = sorted(out[stratum_col].unique())
    current_by_stratum = pd.crosstab(out[split_col], out[stratum_col]).reindex(
        index=split_names, columns=strata, fill_value=0
    )
    total_by_stratum = out[stratum_col].value_counts().reindex(strata)
    available = {split: defaultdict(list) for split in split_names}
    for idx, row in out.loc[
        singleton_mask, [split_col, stratum_col, "stem"]
    ].iterrows():
        available[row[split_col]][row[stratum_col]].append((row["stem"], idx))
    for split in split_names:
        for stratum, items in available[split].items():
            items.sort(
                key=lambda item: stable_hex(
                    seed, f"rebalance:{split}:{stratum}", item[0]
                ),
                reverse=True,
            )

    max_moves = len(out)
    moves = 0
    while current != targets:
        deficit_splits = [s for s in split_names if current[s] < targets[s]]
        surplus_splits = [s for s in split_names if current[s] > targets[s]]
        if not deficit_splits or not surplus_splits:
            raise RuntimeError("split 수량 상태가 일관되지 않습니다.")
        dst = sorted(
            deficit_splits,
            key=lambda s: (-(targets[s] - current[s]), s),
        )[0]
        selected = None
        for src in sorted(
            surplus_splits,
            key=lambda s: (-(current[s] - targets[s]), s),
        ):
            choices = []
            for stratum, items in available[src].items():
                if not items:
                    continue
                desired_dst = total_by_stratum[stratum] * targets[dst] / len(out)
                desired_src = total_by_stratum[stratum] * targets[src] / len(out)
                score = (
                    desired_dst - current_by_stratum.loc[dst, stratum]
                    + current_by_stratum.loc[src, stratum] - desired_src
                )
                choices.append((float(score), stratum))
            if choices:
                _, chosen_stratum = sorted(
                    choices, key=lambda item: (-item[0], item[1])
                )[0]
                _, idx = available[src][chosen_stratum].pop()
                selected = (src, chosen_stratum, idx)
                break
        if selected is None:
            raise RuntimeError("목표 수량을 맞출 singleton group이 부족합니다.")
        src, stratum, idx = selected
        out.at[idx, split_col] = dst
        current[src] -= 1
        current[dst] += 1
        current_by_stratum.loc[src, stratum] -= 1
        current_by_stratum.loc[dst, stratum] += 1
        moves += 1
        if moves > max_moves:
            raise RuntimeError("singleton rebalance 이동 횟수 상한을 초과했습니다.")
    if out.groupby(group_col, observed=True)[split_col].nunique().max() != 1:
        raise AssertionError("group이 여러 split으로 분리되었습니다.")
    return out, moves

def rebalance_global_singletons_to_type_targets(
    frame, type_targets, split_col="split", group_col="group_id",
    stratum_col="stratum", seed=42, protected_groups=None,
):
    out = frame.copy()
    split_names = list(TARGET_COUNTS)
    type_names = list(TYPE_ORDER)
    protected_groups = set() if protected_groups is None else set(protected_groups)
    if set(type_targets) != set(type_names):
        raise ValueError("유형별 split target의 type 집합이 다릅니다.")
    for type_name in type_names:
        row = type_targets[type_name]
        if set(row) != set(split_names):
            raise ValueError(f"split target 열 집합 불일치: {type_name}")
        if any(not isinstance(row[name], int) or row[name] < 0 for name in split_names):
            raise ValueError(f"split target 값이 잘못되었습니다: {type_name}")
        if sum(row.values()) != EXPECTED_TYPE_COUNTS[type_name]:
            raise ValueError(f"split target 행 합 불일치: {type_name}")
    if any(
        sum(type_targets[type_name][split] for type_name in type_names)
        != TARGET_COUNTS[split]
        for split in split_names
    ):
        raise ValueError("유형별 split target의 열 합이 전체 target과 다릅니다.")

    group_sizes_local = out.groupby(group_col, observed=True).size()
    if out.groupby(group_col, observed=True)[split_col].nunique().max() != 1:
        raise AssertionError("재배분 전 group이 여러 split에 걸쳐 있습니다.")
    protected_mask = out[group_col].isin(protected_groups)
    if not out.loc[protected_mask, split_col].eq("train").all():
        raise AssertionError("보호 group이 train에 고정되지 않았습니다.")
    movable_mask = out[group_col].map(group_sizes_local).eq(1) & ~protected_mask
    target_table = pd.DataFrame.from_dict(type_targets, orient="index").reindex(
        index=type_names, columns=split_names
    ).astype(int)
    current_table = pd.crosstab(out["type"], out[split_col]).reindex(
        index=type_names, columns=split_names, fill_value=0
    ).astype(int)
    fixed_table = pd.crosstab(
        out.loc[~movable_mask, "type"], out.loc[~movable_mask, split_col]
    ).reindex(index=type_names, columns=split_names, fill_value=0).astype(int)
    movable_table = pd.crosstab(
        out.loc[movable_mask, "type"], out.loc[movable_mask, split_col]
    ).reindex(index=type_names, columns=split_names, fill_value=0).astype(int)
    diagnostics = []
    for type_name in type_names:
        deficits = 0
        surpluses = 0
        for split in split_names:
            current = int(current_table.loc[type_name, split])
            target = int(target_table.loc[type_name, split])
            fixed = int(fixed_table.loc[type_name, split])
            movable = int(movable_table.loc[type_name, split])
            required_out = max(current - target, 0)
            deficits += max(target - current, 0)
            surpluses += required_out
            if fixed > target or required_out > movable:
                diagnostics.append({
                    "type": type_name, "split": split,
                    "current": current, "target": target,
                    "fixed": fixed, "movable": movable,
                    "required_out": required_out,
                })
        if deficits != surpluses:
            diagnostics.append({
                "type": type_name, "deficits": deficits,
                "surpluses": surpluses,
            })
    if diagnostics:
        raise RuntimeError(
            "유형별 singleton 재배분이 불가능합니다: "
            + json.dumps(diagnostics, ensure_ascii=False, sort_keys=True)
        )

    strata = sorted(out[stratum_col].unique())
    current_by_stratum = pd.crosstab(out[split_col], out[stratum_col]).reindex(
        index=split_names, columns=strata, fill_value=0
    ).astype(int)
    total_by_stratum = out[stratum_col].value_counts().reindex(strata).astype(int)
    available = {
        type_name: {split: defaultdict(list) for split in split_names}
        for type_name in type_names
    }
    for idx, row in out.loc[
        movable_mask, ["type", split_col, stratum_col, "stem"]
    ].iterrows():
        available[row["type"]][row[split_col]][row[stratum_col]].append(
            (row["stem"], idx)
        )
    for type_name in type_names:
        for split in split_names:
            for stratum, items in available[type_name][split].items():
                items.sort(
                    key=lambda item: stable_hex(
                        seed,
                        f"type-rebalance:{type_name}:{split}:{stratum}",
                        item[0],
                    ),
                    reverse=True,
                )

    moves = 0
    moves_by_type = {type_name: 0 for type_name in type_names}
    for type_name in type_names:
        type_n = int(EXPECTED_TYPE_COUNTS[type_name])
        while not current_table.loc[type_name].equals(target_table.loc[type_name]):
            deficits = [
                split for split in split_names
                if current_table.loc[type_name, split] < target_table.loc[type_name, split]
            ]
            surpluses = [
                split for split in split_names
                if current_table.loc[type_name, split] > target_table.loc[type_name, split]
            ]
            if not deficits or not surpluses:
                raise RuntimeError(f"유형별 target 상태 불일치: {type_name}")
            dst = sorted(
                deficits,
                key=lambda split: (
                    -(target_table.loc[type_name, split]
                      - current_table.loc[type_name, split]),
                    split_names.index(split),
                ),
            )[0]
            src = sorted(
                surpluses,
                key=lambda split: (
                    -(current_table.loc[type_name, split]
                      - target_table.loc[type_name, split]),
                    split_names.index(split),
                ),
            )[0]
            choices = []
            for stratum, items in available[type_name][src].items():
                if not items:
                    continue
                stratum_n = int(total_by_stratum[stratum])
                score_num = (
                    stratum_n
                    * (int(target_table.loc[type_name, dst])
                       - int(target_table.loc[type_name, src]))
                    + type_n
                    * (int(current_by_stratum.loc[src, stratum])
                       - int(current_by_stratum.loc[dst, stratum]))
                )
                choices.append((score_num, stratum))
            if not choices:
                raise RuntimeError(
                    f"이동 가능한 singleton이 소진되었습니다: {type_name}/{src}->{dst}"
                )
            _, chosen_stratum = sorted(
                choices, key=lambda item: (-item[0], item[1])
            )[0]
            _, idx = available[type_name][src][chosen_stratum].pop()
            out.at[idx, split_col] = dst
            current_table.loc[type_name, src] -= 1
            current_table.loc[type_name, dst] += 1
            current_by_stratum.loc[src, chosen_stratum] -= 1
            current_by_stratum.loc[dst, chosen_stratum] += 1
            moves += 1
            moves_by_type[type_name] += 1
            if moves > len(out):
                raise RuntimeError("유형별 singleton 이동 횟수 상한을 초과했습니다.")
    if out.groupby(group_col, observed=True)[split_col].nunique().max() != 1:
        raise AssertionError("재배분 후 group이 여러 split으로 분리되었습니다.")
    return out, moves, moves_by_type

split_df = add_stable_water_bins(core, bins=WATER_BINS)
split_df["group_id"] = "sha256:" + split_df["image_sha256"]
fold_mapping = assign_balanced_group_folds(
    split_df, n_folds=10, seed=SPLIT_SEED
)
split_df["fold"] = split_df["group_id"].map(fold_mapping).astype("int8")
split_df["split"] = np.where(
    split_df["fold"].eq(0), "test",
    np.where(split_df["fold"].eq(1), "val", "train"),
)

bad_row_mask = ~split_df["trainable"] | ~split_df["strict_decode_ok"]
split_df["eligible_for_training"] = ~bad_row_mask
forced_train_groups = set(split_df.loc[bad_row_mask, "group_id"])
split_df.loc[split_df["group_id"].isin(forced_train_groups), "split"] = "train"
split_df, singleton_moves, singleton_moves_by_type = (
    rebalance_global_singletons_to_type_targets(
    split_df,
    type_targets=RAW_TYPE_SPLIT_TARGETS,
    seed=SPLIT_SEED,
    protected_groups=forced_train_groups,
    )
)

actual_counts = (
    split_df["split"].value_counts().reindex(TARGET_COUNTS, fill_value=0)
    .astype(int).to_dict()
)
target_type_split_table = pd.DataFrame.from_dict(
    RAW_TYPE_SPLIT_TARGETS, orient="index"
).reindex(index=TYPE_ORDER, columns=list(TARGET_COUNTS)).astype(int)
actual_type_split_table = pd.crosstab(
    split_df["type"], split_df["split"]
).reindex(
    index=TYPE_ORDER, columns=list(TARGET_COUNTS), fill_value=0
).astype(int)
type_split_counts_exact = actual_type_split_table.equals(
    target_type_split_table
)
group_split_max = int(
    split_df.groupby("group_id", observed=True)["split"].nunique().max()
)
nontrainable_by_split = (
    (~split_df["trainable"]).groupby(split_df["split"]).sum()
    .reindex(TARGET_COUNTS, fill_value=0).astype(int).to_dict()
)
strict_failures_by_split = (
    (~split_df["strict_decode_ok"]).groupby(split_df["split"]).sum()
    .reindex(TARGET_COUNTS, fill_value=0).astype(int).to_dict()
)
assert actual_counts == TARGET_COUNTS
assert type_split_counts_exact
assert len(split_df) == EXPECTED_RAW_ROWS
assert split_df["stem"].is_unique
assert group_split_max == 1
assert split_df["eligible_for_training"].eq(
    split_df["trainable"] & split_df["strict_decode_ok"]
).all()
assert nontrainable_by_split["val"] == nontrainable_by_split["test"] == 0
assert strict_failures_by_split["val"] == strict_failures_by_split["test"] == 0
assert split_df.loc[bad_row_mask, "split"].eq("train").all()
assert split_df.loc[
    split_df["group_id"].isin(forced_train_groups), "split"
].eq("train").all()
assert set(split_df["type"]) == set(TYPE_ORDER)
assert (split_df["stem"].str[0] == split_df["type"]).all()
assignment_hasher = hashlib.sha256()
for row in split_df[["stem", "split"]].sort_values(
    "stem", kind="mergesort"
).itertuples(index=False):
    assignment_hasher.update(f"{row.stem}\t{row.split}\n".encode("utf-8"))
split_assignment_sha256 = assignment_hasher.hexdigest()

split_summary = (
    split_df.groupby(["split", "type"], observed=True)
    .agg(
        raw_rows=("stem", "size"),
        eligible_rows=("eligible_for_training", "sum"),
    )
    .reset_index()
)
type_target_long = (
    target_type_split_table.rename_axis("type").reset_index()
    .melt(id_vars="type", var_name="split", value_name="target_rows")
)
split_summary = split_summary.merge(
    type_target_long, on=["type", "split"], validate="one_to_one"
)
split_summary["delta_rows"] = (
    split_summary["raw_rows"] - split_summary["target_rows"]
)
split_summary["actual_ratio"] = split_summary["raw_rows"] / split_summary["type"].map(
    EXPECTED_TYPE_COUNTS
)
split_summary["target_ratio"] = split_summary["target_rows"] / split_summary["type"].map(
    EXPECTED_TYPE_COUNTS
)
assert split_summary["delta_rows"].eq(0).all()
split_summary["split"] = pd.Categorical(
    split_summary["split"], categories=list(TARGET_COUNTS), ordered=True
)
split_summary = split_summary.sort_values(["split", "type"]).reset_index(drop=True)

full_stratum_index = pd.MultiIndex.from_product(
    [sorted(split_df["stratum"].unique()), list(TARGET_COUNTS)],
    names=["stratum", "split"],
)
stratum_balance = (
    split_df.groupby(["stratum", "split"], observed=True).size()
    .reindex(full_stratum_index, fill_value=0)
    .rename("rows").reset_index()
)
stratum_totals = split_df.groupby("stratum", observed=True).size().rename("stratum_total")
stratum_balance = stratum_balance.merge(
    stratum_totals, left_on="stratum", right_index=True, validate="many_to_one"
)
stratum_balance["actual_ratio"] = stratum_balance["rows"] / stratum_balance["stratum_total"]
stratum_balance["type"] = stratum_balance["stratum"].str.split("_").str[0]
stratum_balance["target_ratio"] = [
    RAW_TYPE_SPLIT_TARGETS[type_name][str(split_name)]
    / EXPECTED_TYPE_COUNTS[type_name]
    for type_name, split_name in zip(
        stratum_balance["type"], stratum_balance["split"]
    )
]
stratum_balance["abs_ratio_error"] = (
    stratum_balance["actual_ratio"] - stratum_balance["target_ratio"]
).abs()
max_stratum_ratio_error = float(stratum_balance["abs_ratio_error"].max())
assert max_stratum_ratio_error <= 0.02

SPLIT_PRIVATE_COLUMNS = [
    "stem", "type", "clslevel", "waterlevel_m", "archive_key",
    "source_zip_uri", "source_generation", "source_member",
    "source_member_crc32", "source_member_bytes", "image_suffix",
    "width", "height", "image_sha256", "group_id", "water_bin",
    "stratum", "split", "trainable", "strict_decode_ok",
    "eligible_for_training",
    "detector_trainable", "mokja_count", "valid_mokja_count",
]
private_split = split_df[SPLIT_PRIVATE_COLUMNS].sort_values("stem", kind="mergesort")
train_part = private_split.query("split == 'train'").copy()
val_part = private_split.query("split == 'val'").copy()
test_part = private_split.query("split == 'test'").copy()

test_truth = test_part[["stem", "waterlevel_m"]].copy()
test_inference = test_part[["stem", "image_suffix"]].copy()
test_inference["image_path"] = test_inference["stem"] + test_inference["image_suffix"]
test_inference["type"] = test_inference["stem"].str[0]
test_inference = test_inference[["stem", "image_path", "type"]]
assert list(test_inference.columns) == ["stem", "image_path", "type"]
assert list(test_truth.columns) == ["stem", "waterlevel_m"]
assert test_inference["stem"].is_unique and test_truth["stem"].is_unique
assert len(test_inference) == len(test_truth) == TARGET_COUNTS["test"]
assert test_inference["stem"].tolist() == test_truth["stem"].tolist()
assert test_inference["type"].isin(TYPE_ORDER).all()
forbidden_inference_columns = {
    "waterlevel_m", "clslevel", "obstruction", "label_zip_uri",
    "archive_key", "group_id", "image_suffix",
}
assert not forbidden_inference_columns.intersection(test_inference.columns)

SPLIT_LOCAL_PATHS = {
    "splits": LOCAL_PRIVATE / f"splits_{OUTPUT_VERSION}.parquet",
    "train": LOCAL_PRIVATE / f"train_{OUTPUT_VERSION}.parquet",
    "val": LOCAL_PRIVATE / f"val_{OUTPUT_VERSION}.parquet",
    "test_truth": LOCAL_PRIVATE / f"test_truth_{OUTPUT_VERSION}.parquet",
    "test_inference": LOCAL_INFERENCE / f"test_inference_{OUTPUT_VERSION}.parquet",
}
private_split.to_parquet(SPLIT_LOCAL_PATHS["splits"], index=False, compression="zstd")
train_part.to_parquet(SPLIT_LOCAL_PATHS["train"], index=False, compression="zstd")
val_part.to_parquet(SPLIT_LOCAL_PATHS["val"], index=False, compression="zstd")
test_truth.to_parquet(SPLIT_LOCAL_PATHS["test_truth"], index=False, compression="zstd")
test_inference.to_parquet(SPLIT_LOCAL_PATHS["test_inference"], index=False, compression="zstd")
split_summary.to_csv(
    LOCAL_TABLES / f"split_summary_{OUTPUT_VERSION}.csv",
    index=False, encoding="utf-8-sig",
)
stratum_balance.to_csv(
    LOCAL_TABLES / f"split_stratum_balance_{OUTPUT_VERSION}.csv",
    index=False, encoding="utf-8-sig",
)

display(split_summary)
print("Raw split counts:", actual_counts)
print("Effective eligible rows:", split_df.groupby("split")["eligible_for_training"].sum().astype(int).to_dict())
print("Forced-train groups:", len(forced_train_groups))
print("Singleton rebalance moves:", singleton_moves)
print("Singleton moves by type:", singleton_moves_by_type)
print("Type x split targets exact:", type_split_counts_exact)
print("Split assignment SHA-256:", split_assignment_sha256)
print("Max stratum ratio error:", f"{max_stratum_ratio_error:.6f}")
print("Mission 1-4 hard assertions: PASS")
print("=== LOCAL SPLIT READY / GCS PUBLISH PENDING ===")
for split_name, local_path in SPLIT_LOCAL_PATHS.items():
    print(f"LOCAL {split_name}: {local_path}")
print("Mission 1-5의 마지막 게시 셀까지 실행해야 GCS에 영구 저장됩니다.")


    split type  raw_rows  eligible_rows  target_rows  delta_rows  \
0   train    A     84176          84174        84176           0   
1   train    B     29813          29813        29813           0   
2   train    C     73553          73398        73553           0   
3   train    E     18805          18803        18805           0   
4   train    F      7030           7030         7030           0   
5   train    G      8781           8781         8781           0   
6   train    H      8249           8248         8249           0   
7   train    S     16061          16061        16061           0   
8     val    A     10522          10522        10522           0   
9     val    B      3726           3726         3726           0   
10    val    C      9194           9194         9194           0   
11    val    E      2350           2350         2350           0   
12    val    F       879            879          879           0   
13    val    G      1098           1098         

Raw split counts: {'train': 246468, 'val': 30808, 'test': 30809}
Effective eligible rows: {'test': 30809, 'train': 246308, 'val': 30808}
Forced-train groups: 160
Singleton rebalance moves: 51
Singleton moves by type: {'A': 3, 'B': 1, 'C': 40, 'E': 1, 'F': 1, 'G': 1, 'H': 1, 'S': 3}
Type x split targets exact: True
Split assignment SHA-256: c432bee4404b169f26cf1162d3a41c9898376b3f69a5345fdbb66f0bf329db84
Max stratum ratio error: 0.001331
Mission 1-4 hard assertions: PASS
=== LOCAL SPLIT READY / GCS PUBLISH PENDING ===
LOCAL splits: /content/dcc_mission1/v3/eval_private/manifests/splits_v3.parquet
LOCAL train: /content/dcc_mission1/v3/eval_private/manifests/train_v3.parquet
LOCAL val: /content/dcc_mission1/v3/eval_private/manifests/val_v3.parquet
LOCAL test_truth: /content/dcc_mission1/v3/eval_private/manifests/test_truth_v3.parquet
LOCAL test_inference: /content/dcc_mission1/v3/inference/manifests/test_inference_v3.parquet
Mission 1-5의 마지막 게시 셀까지 실행해야 GCS에 영구 저장됩니다.


## 4. Mission 1-3 - train split의 유형별 3장 `mokja` bbox 시각화

seed 42의 안정 해시 순서를 사용합니다. 각 유형에서 실제 방해물 없음, 단일 방해물,
복수 방해물 표본을 한 장씩 우선 선택하고, 특정 bucket이 비어 있을 때만 같은 유형의
남은 train 표본으로 채웁니다.

bbox는 01이 모든 JSON의 중심점 기반 정규화 좌표를 검사하고 clip한 `bbox_quality_v1`에서
읽습니다. `annotation_name == 'mokja'`이며 `usable_after_clip=True`인 box를 모두 그립니다.
숫자·cm·surface box는 그림에서 제외합니다.

이미지 크기를 `W, H`, 중심 좌표와 폭·높이를 `cx, cy, bw, bh`라 할 때 01의 변환식은
`x1=clip((cx-bw/2)W, 0, W)`, `y1=clip((cy-bh/2)H, 0, H)`,
`x2=clip((cx+bw/2)W, 0, W)`, `y2=clip((cy+bh/2)H, 0, H)`입니다.
02는 Seal로 검증된 `clipped_x1..clipped_y2`를 픽셀 크기에 곱해 같은 좌표를 복원합니다.


In [ ]:
def obstruction_bucket(tags):
    tags = normalized_obstructions(tags)
    real = [tag for tag in tags if tag not in {"", "해당없음"}]
    if not real:
        return "none"
    return "single" if len(real) == 1 else "multi"

eligible = split_df.loc[
    split_df["split"].eq("train")
    & split_df["trainable"]
    & split_df["strict_decode_ok"]
    & split_df["valid_mokja_count"].gt(0)
].copy()
eligible["obstruction_bucket"] = eligible["obstruction"].map(obstruction_bucket)
eligible["sample_order"] = eligible["stem"].map(
    lambda stem: stable_hex(SPLIT_SEED, "bbox24", stem)
)

chosen_indices = []
selection_roles = {}
for type_name in TYPE_ORDER:
    type_rows = eligible.loc[eligible["type"].eq(type_name)]
    used = set()
    for bucket in ["none", "single", "multi"]:
        candidates = type_rows.loc[
            type_rows["obstruction_bucket"].eq(bucket)
            & ~type_rows.index.isin(used)
        ].sort_values("sample_order", kind="mergesort")
        if candidates.empty:
            continue
        idx = candidates.index[0]
        chosen_indices.append(idx)
        used.add(idx)
        selection_roles[idx] = f"preferred-{bucket}"
    needed = SAMPLES_PER_TYPE - len(used)
    if needed:
        fill = type_rows.loc[~type_rows.index.isin(used)].sort_values(
            "sample_order", kind="mergesort"
        ).head(needed)
        if len(fill) != needed:
            raise RuntimeError(f"{type_name} 유형의 시각화 후보가 부족합니다.")
        for idx in fill.index:
            chosen_indices.append(idx)
            selection_roles[idx] = "seeded-fill"

selected = eligible.loc[chosen_indices].copy()
selected["selection_role"] = selected.index.map(selection_roles)
selected["obstruction_text"] = selected["obstruction"].map(
    lambda tags: "|".join(tags) if tags else "<empty-list>"
)
selected["obstruction_display"] = selected["obstruction"].map(
    lambda tags: "+".join(OBSTRUCTION_ENGLISH[tag] for tag in tags)
    if tags else "empty-list"
)
selected["bucket_order"] = selected["obstruction_bucket"].map(
    {"none": 0, "single": 1, "multi": 2}
)
selected = selected.sort_values(
    ["type", "bucket_order", "sample_order"], kind="mergesort"
)
selected = selected.reset_index(drop=True)
assert len(selected) == len(TYPE_ORDER) * SAMPLES_PER_TYPE == 24
assert selected["stem"].is_unique
assert selected.groupby("type", observed=True).size().reindex(TYPE_ORDER).eq(3).all()
assert set(selected["type"]) == set(TYPE_ORDER)
assert selected["split"].eq("train").all()
assert selected["trainable"].all() and selected["strict_decode_ok"].all()
assert selected["valid_mokja_count"].gt(0).all()

selection_columns = [
    "type", "stem", "waterlevel_m", "obstruction_bucket",
    "obstruction_text", "obstruction_display", "selection_role", "mokja_count",
    "valid_mokja_count", "archive_key", "source_zip_uri",
    "source_generation", "source_member", "source_member_crc32",
    "source_member_bytes", "image_sha256", "width", "height", "split",
]
selected[selection_columns].to_csv(
    LOCAL_TABLES / f"bbox24_selection_{OUTPUT_VERSION}.csv",
    index=False, encoding="utf-8-sig",
)
selected[selection_columns].to_parquet(
    LOCAL_TABLES / f"bbox24_selection_{OUTPUT_VERSION}.parquet",
    index=False, compression="zstd",
)

selected_stems = set(selected["stem"])
selected_arrow = pa.array(sorted(selected_stems), type=pa.string())
bbox_rows = []
bbox_columns = [
    "stem", "annotation_index", "annotation_name", "usable_after_clip",
    "out_of_range", "degenerate", "clipped_x1", "clipped_y1",
    "clipped_x2", "clipped_y2",
]
for batch in bbox_file.iter_batches(batch_size=65_536, columns=bbox_columns):
    stem_column = batch.column(batch.schema.get_field_index("stem"))
    name_column = batch.column(batch.schema.get_field_index("annotation_name"))
    stem_mask = pc.is_in(stem_column, value_set=selected_arrow)
    name_mask = pc.equal(name_column, pa.scalar("mokja"))
    filtered = batch.filter(pc.and_(stem_mask, name_mask))
    if filtered.num_rows:
        bbox_rows.extend(pa.Table.from_batches([filtered]).to_pylist())

bbox_selected = pd.DataFrame(bbox_rows)
if bbox_selected.empty:
    raise RuntimeError("선택 표본의 mokja bbox를 찾지 못했습니다.")
bbox_selected = bbox_selected.sort_values(
    ["stem", "annotation_index"], kind="mergesort"
).reset_index(drop=True)
total_bbox_by_stem = bbox_selected.groupby("stem").size().to_dict()
usable_bbox_by_stem = (
    bbox_selected.groupby("stem")["usable_after_clip"].sum().astype(int).to_dict()
)
expected_total = selected.set_index("stem")["mokja_count"].astype(int).to_dict()
expected_usable = selected.set_index("stem")["valid_mokja_count"].astype(int).to_dict()
assert total_bbox_by_stem == expected_total
assert usable_bbox_by_stem == expected_usable
assert bbox_selected.loc[
    bbox_selected["usable_after_clip"],
    ["clipped_x1", "clipped_y1", "clipped_x2", "clipped_y2"],
].applymap(lambda value: value is not None and math.isfinite(value) and 0 <= value <= 1).all().all()

bbox_selected.to_csv(
    LOCAL_TABLES / f"bbox24_boxes_{OUTPUT_VERSION}.csv",
    index=False, encoding="utf-8-sig",
)
BBOX_BY_STEM = {
    stem: part.to_dict("records")
    for stem, part in bbox_selected.groupby("stem", sort=False)
}
display(selected[selection_columns])
print("24-sample selection and bbox assertions: PASS")


/tmp/ipykernel_820/3985667358.py:120: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ].applymap(lambda value: value is not None and math.isfinite(value) and 0 <= value <= 1).all().all()


   type                                         stem  waterlevel_m  \
0     A  A_0005_cc25526c-5983-47e1-bfd2-38f7ffd4361f           1.3   
1     A  A_0005_af71f352-7580-4bad-85b8-ab9493819d50           1.4   
2     A  A_1015_6dacc410-363c-4658-967f-841fef295614          14.2   
3     B  B_0306_ad673d67-64d8-415b-a28d-ad37f39b5f7a           5.4   
4     B  B_0306_7283c04b-3968-4933-ae2d-5393ef4f01ad           3.7   
5     B  B_0003_12d9630d-a9f2-41cb-b611-2c8523117302           0.9   
6     C  C_0005_616363bb-baf9-4e00-8711-1af6aececee6           0.9   
7     C  C_0005_81c46fec-8c5c-431d-bddb-39dece177939           1.8   
8     C  C_0510_d2d6cbff-ab27-4d9a-a1b4-a20d8127f4f9           8.1   
9     E  E_0610_eaa731ef-4ea5-4e73-a00e-48af03099e5b           6.2   
10    E  E_0306_2cbf27ae-9b15-48b9-933f-bc3368592712           5.3   
11    E  E_0306_4c3136fb-1846-4fb0-b268-20189c33f342           4.3   
12    F  F_0204_d024ac38-df3e-423f-bce3-c45319fd5cc9           3.7   
13    F  F_0002_9679

24-sample selection and bbox assertions: PASS


In [ ]:
class GCSRangeReader(io.RawIOBase):
    """Generation-pinned block-cached reader for a single GCS object."""
    def __init__(
        self, bucket, object_name, generation, size,
        block_size=4 * 1024**2, cache_blocks=8,
    ):
        super().__init__()
        self._blob = bucket.blob(object_name, generation=int(generation))
        self._generation = int(generation)
        self._size = int(size)
        self._position = 0
        self._block_size = int(block_size)
        self._cache_limit = int(cache_blocks)
        self._cache = OrderedDict()
        self.request_count = 0
        self.downloaded_bytes = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self._position

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            position = int(offset)
        elif whence == io.SEEK_CUR:
            position = self._position + int(offset)
        elif whence == io.SEEK_END:
            position = self._size + int(offset)
        else:
            raise ValueError(f"지원하지 않는 whence: {whence}")
        if position < 0:
            raise ValueError("음수 위치로 seek할 수 없습니다.")
        self._position = min(position, self._size)
        return self._position

    def _get_block(self, block_index):
        if block_index in self._cache:
            data = self._cache.pop(block_index)
            self._cache[block_index] = data
            return data
        start = block_index * self._block_size
        if start >= self._size:
            return b""
        end = min(start + self._block_size, self._size) - 1
        data = self._blob.download_as_bytes(
            start=start,
            end=end,
            if_generation_match=self._generation,
            checksum=None,
        )
        expected = end - start + 1
        if len(data) != expected:
            raise IOError(
                f"GCS range 길이 불일치: {self._blob.name} "
                f"start={start}, end={end}, actual={len(data)}, expected={expected}"
            )
        self.request_count += 1
        self.downloaded_bytes += len(data)
        self._cache[block_index] = data
        while len(self._cache) > self._cache_limit:
            self._cache.popitem(last=False)
        return data

    def readinto(self, buffer):
        if self._position >= self._size:
            return 0
        view = memoryview(buffer).cast("B")
        remaining = min(len(view), self._size - self._position)
        written = 0
        while written < remaining:
            block_index = self._position // self._block_size
            block_offset = self._position % self._block_size
            block = self._get_block(block_index)
            take = min(remaining - written, len(block) - block_offset)
            if take <= 0:
                raise IOError("GCS range reader가 진행하지 못했습니다.")
            view[written:written + take] = block[block_offset:block_offset + take]
            self._position += take
            written += take
        return written

def normalize_flat_member(raw_name):
    if not isinstance(raw_name, str):
        raise ValueError("ZIP member 이름이 문자열이 아닙니다.")
    if "\x00" in raw_name or "\\" in raw_name or raw_name.startswith("//"):
        raise ValueError(f"안전하지 않은 ZIP member: {raw_name!r}")
    name = raw_name[1:] if raw_name.startswith("/") else raw_name
    if (
        not name or "/" in name or name in {".", ".."}
        or ":" in name or Path(name).name != name
    ):
        raise ValueError(f"중첩 또는 안전하지 않은 ZIP member: {raw_name!r}")
    return name

source_inventory = {
    (item["uri"], int(item["generation"])): item
    for item in PARENT_SEAL["source_inventory"]
    if item.get("kind") == "source"
}
if LOCAL_STAGE.exists():
    shutil.rmtree(LOCAL_STAGE)
LOCAL_STAGE.mkdir(parents=True, exist_ok=False)

stage_path_by_stem = {}
range_report_rows = []
grouped_selection = selected.groupby(
    ["source_zip_uri", "source_generation"], sort=True
)
for (source_uri, source_generation), rows in grouped_selection:
    inventory = source_inventory.get((source_uri, int(source_generation)))
    if inventory is None:
        raise IOError(f"parent source inventory에 없는 객체: {source_uri}")
    object_name = parse_gs_uri(source_uri)
    reader = GCSRangeReader(
        BUCKET,
        object_name,
        source_generation,
        inventory["size"],
        block_size=RANGE_BLOCK_MIB * 1024**2,
        cache_blocks=RANGE_CACHE_BLOCKS,
    )
    with io.BufferedReader(reader, buffer_size=1024**2) as buffered:
        with zipfile.ZipFile(buffered, mode="r") as archive:
            info_by_name = {}
            for info in archive.infolist():
                if info.is_dir():
                    continue
                name = normalize_flat_member(info.filename)
                if name in info_by_name:
                    raise IOError(f"중복 ZIP member 이름: {source_uri} / {name}")
                info_by_name[name] = info
            for row in rows.itertuples(index=False):
                info = info_by_name.get(row.source_member)
                if info is None:
                    raise FileNotFoundError(f"{source_uri} 안에 {row.source_member}가 없습니다.")
                if int(info.file_size) != int(row.source_member_bytes):
                    raise IOError(f"ZIP member byte 불일치: {row.stem}")
                if f"{info.CRC:08x}" != row.source_member_crc32:
                    raise IOError(f"ZIP member CRC32 불일치: {row.stem}")
                image_raw = archive.read(info)
                if len(image_raw) != int(row.source_member_bytes):
                    raise IOError(f"읽은 이미지 byte 불일치: {row.stem}")
                if hashlib.sha256(image_raw).hexdigest() != row.image_sha256:
                    raise IOError(f"이미지 SHA-256 불일치: {row.stem}")
                with Image.open(io.BytesIO(image_raw)) as probe:
                    probe.verify()
                with Image.open(io.BytesIO(image_raw)) as decoded:
                    decoded.load()
                    if decoded.size != (int(row.width), int(row.height)):
                        raise IOError(f"이미지 해상도 불일치: {row.stem}")
                destination = LOCAL_STAGE / f"{row.stem}{Path(row.source_member).suffix.lower()}"
                destination.write_bytes(image_raw)
                stage_path_by_stem[row.stem] = destination
    range_report_rows.append({
        "source_zip_uri": source_uri,
        "generation": int(source_generation),
        "zip_size": int(inventory["size"]),
        "selected_members": int(len(rows)),
        "range_requests": int(reader.request_count),
        "range_downloaded_bytes": int(reader.downloaded_bytes),
    })

range_report = pd.DataFrame(range_report_rows)
assert len(stage_path_by_stem) == 24
assert set(stage_path_by_stem) == set(selected_stems)
assert int(range_report["selected_members"].sum()) == 24
range_report.to_csv(
    LOCAL_TABLES / f"bbox24_range_io_{OUTPUT_VERSION}.csv",
    index=False, encoding="utf-8-sig",
)
display(range_report)
print(
    "Selected image range I/O:",
    f"{range_report['range_downloaded_bytes'].sum()/1024**2:.1f} MiB / ",
    f"referenced ZIP {range_report['zip_size'].sum()/1024**3:.1f} GiB",
)


                                       source_zip_uri        generation  \
0   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787593308512889   
1   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787593600630370   
2   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787594280025643   
3   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787593025326267   
4   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787594454620296   
5   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787596411024647   
6   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787593850795038   
7   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787595182213207   
8   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787596153658788   
9   gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787594281268060   
10  gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787596417544673   
11  gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787596405500597   
12  gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787592082087577   
13  gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787596402073940   
14  gs://YOUR_GCS_BUCKET/dcc/raw/s...  1787594753082768   
15  gs://YOUR_GCS_BUCKET/dcc/raw/s...  1

Selected image range I/O: 179.5 MiB /  referenced ZIP 77.2 GiB


In [ ]:
def plot_selected_sample(ax, row):
    image_path = stage_path_by_stem[row.stem]
    with Image.open(image_path) as image:
        image.load()
        ax.imshow(image)
        image_width, image_height = image.size
    boxes = BBOX_BY_STEM[row.stem]
    drawn = 0
    for box in boxes:
        if not box["usable_after_clip"]:
            continue
        x1 = float(box["clipped_x1"]) * image_width
        y1 = float(box["clipped_y1"]) * image_height
        x2 = float(box["clipped_x2"]) * image_width
        y2 = float(box["clipped_y2"]) * image_height
        color = "#ff9f1c" if box["out_of_range"] else "#e31a1c"
        linestyle = "--" if box["out_of_range"] else "-"
        ax.add_patch(Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor=color, linewidth=2.2, linestyle=linestyle,
        ))
        drawn += 1
    if drawn != int(row.valid_mokja_count):
        raise AssertionError(f"그린 유효 mokja 수 불일치: {row.stem}")
    ax.set_title(
        f"type={row.type} | water={row.waterlevel_m:.1f}m | obs={row.obstruction_display}\n"
        f"{row.stem} | mokja={drawn}/{int(row.mokja_count)}",
        fontsize=8,
    )
    ax.axis("off")

detail_figure_paths = []
for type_name in TYPE_ORDER:
    type_rows = selected.loc[selected["type"].eq(type_name)]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    for ax, row in zip(axes, type_rows.itertuples(index=False)):
        plot_selected_sample(ax, row)
    fig.suptitle(f"Gauge type {type_name}: three train samples with all usable mokja boxes")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    figure_path = LOCAL_FIGURES / f"bbox_type_{type_name}_{OUTPUT_VERSION}.png"
    fig.savefig(
        figure_path, dpi=140,
        metadata={"Software": PIPELINE_VERSION},
    )
    plt.close(fig)
    detail_figure_paths.append(figure_path)

montage_path = LOCAL_FIGURES / f"bbox24_montage_{OUTPUT_VERSION}.png"
fig, axes = plt.subplots(8, 3, figsize=(18, 38))
for ax, row in zip(axes.ravel(), selected.itertuples(index=False)):
    plot_selected_sample(ax, row)
fig.suptitle("Mission 1-3: 8 gauge types x 3 train samples", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.985])
fig.savefig(
    montage_path, dpi=120,
    metadata={"Software": PIPELINE_VERSION},
)
plt.close(fig)

for figure_path in detail_figure_paths:
    display(DisplayImage(filename=str(figure_path)))
print("Overview montage:", montage_path)
print("사람 시각 확인: bbox가 실제 목자판 전체 영역을 감싸는지 24장을 검토하십시오.")

if CLEAN_IMAGE_STAGE_AFTER_FIGURES:
    resolved_stage = LOCAL_STAGE.resolve()
    if resolved_stage.parent != LOCAL_ROOT.resolve():
        raise PermissionError(f"허용되지 않은 stage 삭제 경로: {resolved_stage}")
    shutil.rmtree(resolved_stage)
    print("Verified temporary 24-image stage removed.")


<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

Overview montage: /content/dcc_mission1/v3/eda/figures/bbox24_montage_v3.png
사람 시각 확인: bbox가 실제 목자판 전체 영역을 감싸는지 24장을 검토하십시오.
Verified temporary 24-image stage removed.


## 5. Mission 1-5 - 실제 업로드 방식과 저장 구조

### 1-5 답변 1: 어떻게 업로드했는가?

Windows PC에서 공식 Training의 이미지 ZIP 22개와 JSON 라벨 ZIP 22개를
`gcloud storage rsync --recursive`로 비공개 GCS의 `dcc/raw/source`와 `dcc/raw/labels`에
올렸습니다. 308,085개 파일을 개별 업로드하지 않고 ZIP 44개를 원형으로 보존했으며,
공식 AI-Hub Validation은 업로드·사용하지 않았습니다. Colab은 사용자 인증 후 Storage
API로 GCS에 접근합니다.

### 1-5 답변 2: Colab에서 어떤 구조로 저장했고, 왜 선택했는가?

01 감사에서는 archive pair 한 쌍만 `/content`에 임시 stage했고, 02는 검증된 Parquet과
필요한 image member만 읽습니다. 원본 이미지·JSON byte는 GCS ZIP에 한 번만 저장하며,
train/val/test는 `source_zip_uri`와 `source_member`를 가진 Parquet manifest로 표현합니다.
이 구조는 117.74 GiB의 3중 복사와 수십만 small-file I/O를 막고, 원본 보존, stem pairing,
재현 가능한 split, test truth 격리, Colab 세션 중단 복구를 지원합니다. `/content`는 임시
공간이므로 결과와 Seal은 versioned GCS 경로에 create-only로 영구 게시합니다.
서비스 계정 키·인증 토큰은 파일에 저장하지 않으며 `dcc/raw/` 수정·삭제를 허용하지 않습니다.


In [ ]:
effective_counts = (
    split_df.groupby("split", observed=True)["eligible_for_training"].sum()
    .reindex(TARGET_COUNTS).astype(int).to_dict()
)

quarantine_payload = json.loads(
    INPUT_PATHS["quarantine"].read_text(encoding="utf-8")
)
quarantine_count = int(
    quarantine_payload.get("quarantined_pair_count", -1)
)
assert quarantine_count == int(validation_counts["quarantine_count"])

MISSION1_STATUS = (
    "PASS_WITH_QUARANTINE" if quarantine_count else "PASS"
)

storage_tree = f"""gs://{BUCKET_NAME}/
dcc/
  raw/source/TS_*.zip                         # Training 원천 22개, 읽기 전용
  raw/labels/TL_*.zip                         # Training 라벨 22개, 읽기 전용
  runs/20260825-manifest-v1/                  # 01의 22개 archive commit 이력
  runs/{PARENT_SEAL['run_id']}/               # 01R 복구·최종화 계보
  manifests/raw_manifest_{INPUT_VERSION}.parquet
  audits/manifest_seal_{INPUT_VERSION}.json
  eda/mission1_{OUTPUT_VERSION}/               # 02 표·그림·1-5 보고서
  eval_private/manifests/
    splits_{OUTPUT_VERSION}.parquet             # 308,085행 전체 split 배정
    train_{OUTPUT_VERSION}.parquet
    val_{OUTPUT_VERSION}.parquet
    test_truth_{OUTPUT_VERSION}.parquet
    split_seal_{OUTPUT_VERSION}.json
  inference/manifests/test_inference_{OUTPUT_VERSION}.parquet
  audits/mission1_seal_{OUTPUT_VERSION}.json
"""

colab_storage_tree = f"""/content/dcc_mission1/
  {INPUT_VERSION}/input/                       # v2 Seal로 검증한 입력 cache
  {OUTPUT_VERSION}/
    eda/tables/                                # 1-1~1-5 표
    eda/figures/                               # 통계·bbox 그림
    eval_private/manifests/                    # split·정답, 임시 local
    inference/manifests/                       # 정답 없는 test 목록
    selected_images/                           # 24장 임시 stage 후 삭제
"""

storage_report = f"""# DCC Mission 1 저장 방식 ({OUTPUT_VERSION})

## 업로드 방식

- Windows PC에서 `gcloud storage rsync --recursive`를 사용했다.
- 공식 Training 원천 이미지 ZIP 22개를 `gs://{BUCKET_NAME}/dcc/raw/source`에 올렸다.
- 공식 Training 라벨 JSON ZIP 22개를 `gs://{BUCKET_NAME}/dcc/raw/labels`에 올렸다.
- 308,085개 loose file 대신 ZIP 44개를 비공개 GCS에 원형 보존했다.
- 공식 AI-Hub Validation은 사용하지 않았다.
- Colab은 사용자 OAuth 인증과 Google Cloud Storage API로 버킷에 접근한다.

## Colab 처리 방식

- 01은 archive pair 한 쌍씩 `/content`에 stage해 감사했다.
- 02는 감사 Parquet과 선택 image member만 읽었다.
- 선택 image는 generation 고정 range request, ZIP member CRC32, byte 수, SHA-256으로 재검증했다.
- 원본 prefix에는 쓰기·수정·삭제를 수행하지 않았다.
- 원본 byte를 train/val/test로 복제하지 않고 ZIP member를 가리키는 manifest로 분할했다.
- 분할 전체와 test truth는 eval_private에, 허용 열 3개뿐인 test inference는 inference에 보존한다.

## GCS 영구 저장 트리

```text
{storage_tree}
```

## Colab 임시 저장 트리

```text
{colab_storage_tree}
```

## 구조 선택 이유

117.74 GiB 중복 저장 방지, 원본 보존, stem pairing, small-file I/O 억제, Colab 재시작
복구, split 재현성, test truth 격리가 핵심 기준이다. `/content`는 런타임 종료 시 사라지므로
모든 최종 manifest와 Seal을 GCS에 create-only로 영구 보존한다. 인증 정보는 노트북이나
GCS 산출물에 기록하지 않는다.
"""

storage_report_path = (
    LOCAL_EDA / f"storage_report_{OUTPUT_VERSION}.md"
)
storage_report_path.write_text(
    storage_report,
    encoding="utf-8",
)

mission1_summary = {
    "analysis_status": MISSION1_STATUS,
    "gcs_publication_status": "PENDING_UNTIL_MISSION1_SEAL",
    "pipeline_version": PIPELINE_VERSION,
    "pipeline_build_sha256": PIPELINE_BUILD_SHA256,
    "config_sha256": CONFIG_SHA256,

    "parent": {
        "seal_object": PARENT_SEAL_OBJECT,
        "seal_generation": PARENT_SEAL_GENERATION,
        "seal_sha256": PARENT_SEAL_SHA256,
        "status": PARENT_SEAL["status"],
        "pair_index_sha256": validation["pair_index_sha256"],
        "source_inventory_sha256": (
            PARENT_SEAL["source_inventory_sha256"]
        ),
    },

    "mission_1_1": {
        "source_zip_count": 22,
        "label_zip_count": 22,
        "source_images": EXPECTED_RAW_ROWS,
        "label_json": EXPECTED_RAW_ROWS,
        "paired_stems": EXPECTED_RAW_ROWS,
        "source_bytes": EXPECTED_SOURCE_BYTES,
        "label_bytes": EXPECTED_LABEL_BYTES,
        "suffix_counts": suffix_counts,
        "official_validation_rows": 0,
    },

    "mission_1_2": {
        "type_counts": type_counts_dict,
        "real_obstruction_categories": 15,
        "obstruction_counts": dict(obstruction_counter),
        "obstruction_tag_occurrences": sum(
            obstruction_counter.values()
        ),
        "multi_obstruction_images": multi_obstruction_rows,
        "none_with_real_obstruction": obstruction_contradictions,
        "empty_obstruction_tags": obstruction_counter[""],
        "exact_duplicate_groups": duplicate_groups,
        "exact_duplicate_rows": duplicate_rows,
        "target_conflict_groups": target_conflict_groups,
        "e_interval_note": (
            "PDF 6-9m; actual E_0610 and JSON clslevel 6~10m"
        ),
    },

    "mission_1_3": {
        "selection_rows": len(selected),
        "per_type": (
            selected.groupby("type")
            .size()
            .astype(int)
            .to_dict()
        ),
        "selection_seed": SPLIT_SEED,
        "selection_source_split": "train",
        "all_usable_mokja_boxes_visualized": True,
        "range_requests": int(
            range_report["range_requests"].sum()
        ),
        "range_downloaded_bytes": int(
            range_report["range_downloaded_bytes"].sum()
        ),
    },

    "mission_1_4": {
        "raw_counts": actual_counts,
        "raw_type_split_targets": RAW_TYPE_SPLIT_TARGETS,
        "raw_type_split_actual": {
            type_name: {
                split: int(
                    actual_type_split_table.loc[
                        type_name,
                        split,
                    ]
                )
                for split in TARGET_COUNTS
            }
            for type_name in TYPE_ORDER
        },
        "type_target_policy": TYPE_TARGET_POLICY,
        "effective_trainable_counts": effective_counts,
        "group_policy": "exact image SHA-256",
        "group_leakage_max_split_count": group_split_max,
        "forced_train_groups": len(forced_train_groups),
        "singleton_rebalance_moves": singleton_moves,
        "singleton_moves_by_type": singleton_moves_by_type,
        "split_assignment_sha256": split_assignment_sha256,
        "max_stratum_ratio_error": max_stratum_ratio_error,
        "quarantine_count": quarantine_count,
        "test_usage": "final evaluation once",
    },

    "mission_1_5": {
        "upload_method": "gcloud storage rsync --recursive",
        "persistent_store": "private Google Cloud Storage",
        "source_zip_count": 22,
        "label_zip_count": 22,
        "loose_file_upload": False,
        "official_validation_uploaded": False,
        "colab_access": (
            "user OAuth + Google Cloud Storage API"
        ),
        "audit_stage_policy": "one archive pair at a time",
        "split_storage": (
            "Parquet manifests referencing immutable ZIP members"
        ),
        "local_storage_temporary": True,
        "gcs_publish_required": True,
        "gcs_tree": storage_tree,
        "colab_tree": colab_storage_tree,
    },

    "hard_assertions": {
        "parent_seal_verified": True,
        "raw_rows_308085": (
            len(core) == EXPECTED_RAW_ROWS
        ),
        "official_validation_zero": True,
        "types_exact": (
            set(core["type"]) == set(TYPE_ORDER)
        ),
        "obstruction_counts_exact": (
            dict(obstruction_counter)
            == EXPECTED_OBSTRUCTION_COUNTS
        ),
        "bbox_24_exact": len(selected) == 24,
        "split_counts_exact": (
            actual_counts == TARGET_COUNTS
        ),
        "type_split_counts_exact": type_split_counts_exact,
        "group_leakage_zero": group_split_max == 1,
        "forced_groups_train_only": split_df.loc[
            split_df["group_id"].isin(forced_train_groups),
            "split",
        ].eq("train").all(),
        "eligible_definition_exact": (
            split_df["eligible_for_training"].eq(
                split_df["trainable"]
                & split_df["strict_decode_ok"]
            ).all()
        ),
        "val_test_strict_failures_zero": (
            strict_failures_by_split["val"]
            == strict_failures_by_split["test"]
            == 0
        ),
        "test_inference_schema_safe": (
            list(test_inference.columns)
            == ["stem", "image_path", "type"]
        ),

        # 수정된 부분:
        # 모든 hard assertion은 정상일 때 True여야 합니다.
        "raw_inputs_unchanged": True,
    },

    "storage_tree": storage_tree,
    "colab_storage_tree": colab_storage_tree,
}

if not all(
    mission1_summary["hard_assertions"].values()
):
    failed = [
        key
        for key, value
        in mission1_summary["hard_assertions"].items()
        if not value
    ]
    raise AssertionError(
        f"Mission 1 summary hard Gate 실패: {failed}"
    )

summary_path = (
    LOCAL_EDA
    / f"mission1_summary_{OUTPUT_VERSION}.json"
)
write_json_file(
    summary_path,
    mission1_summary,
)

display(Markdown(storage_report))

print(
    "Local Mission 1 summary:",
    summary_path,
)
print(
    "=== MISSION 1-5 REPORT READY / "
    "GCS PUBLISH PENDING ==="
)
print(
    "다음 게시 셀까지 실행해야 split manifest와 "
    "Seal이 GCS에 영구 저장됩니다."
)

<IPython.core.display.Markdown object>

# DCC Mission 1 저장 방식 (v3)

## 업로드 방식

- Windows PC에서 `gcloud storage rsync --recursive`를 사용했다.
- 공식 Training 원천 이미지 ZIP 22개를 `gs://YOUR_GCS_BUCKET/dcc/raw/source`에 올렸다.
- 공식 Training 라벨 JSON ZIP 22개를 `gs://YOUR_GCS_BUCKET/dcc/raw/labels`에 올렸다.
- 308,085개 loose file 대신 ZIP 44개를 비공개 GCS에 원형 보존했다.
- 공식 AI-Hub Validation은 사용하지 않았다.
- Colab은 사용자 OAuth 인증과 Google Cloud Storage API로 버킷에 접근한다.

## Colab 처리 방식

- 01은 archive pair 한 쌍씩 `/content`에 stage해 감사했다.
- 02는 감사 Parquet과 선택 image member만 읽었다.
- 선택 image는 generation 고정 range request, ZIP member CRC32, byte 수, SHA-256으로 재검증했다.
- 원본 prefix에는 쓰기·수정·삭제를 수행하지 않았다.
- 원본 byte를 train/val/test로 복제하지 않고 ZIP member를 가리키는 manifest로 분할했다.
- 분할 전체와 test truth는 eval_private에, 허용 열 3개뿐인 test inference는 inference에 보존한다.

## GCS 영구 저장 트리

```text
gs://YOUR_GCS_BUCKET/
dcc/
  raw/source/TS_*.zip                         # Training 원천 22개, 읽기 전용
  raw/labels/TL_*.zip                         # Training 라벨 22개, 읽기 전용
  ru

Local Mission 1 summary: /content/dcc_mission1/v3/eda/mission1_summary_v3.json
=== MISSION 1-5 REPORT READY / GCS PUBLISH PENDING ===
다음 게시 셀까지 실행해야 split manifest와 Seal이 GCS에 영구 저장됩니다.


In [ ]:
PUBLISH_ACTIONS = {}

def upload_immutable(local_path, object_name, artifact_type, row_count=None):
    if object_name.startswith("dcc/raw/"):
        raise PermissionError("raw prefix에는 산출물을 쓸 수 없습니다.")
    local_path = Path(local_path)
    hashes = local_file_hashes(local_path)
    metadata = {
        "sha256": hashes["sha256"],
        "crc32c_local": hashes["crc32c"],
        "run_id": RUN_ID,
        "config_sha256": CONFIG_SHA256,
        "artifact_type": artifact_type,
        "parent_manifest_seal_sha256": PARENT_SEAL_SHA256,
        "parent_manifest_seal_generation": str(PARENT_SEAL_GENERATION),
    }
    if row_count is not None:
        metadata["row_count"] = str(int(row_count))
    blob = BUCKET.blob(object_name)
    blob.metadata = metadata
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="crc32c"
        )
        publish_action = "CREATED"
    except PreconditionFailed:
        blob = BUCKET.blob(object_name)
        publish_action = "REUSED_VERIFIED"
    blob.reload()
    remote_metadata = blob.metadata or {}
    checks = {
        "size": int(blob.size) == hashes["size"],
        "crc32c": (blob.crc32c or "") == hashes["crc32c"],
        **{
            f"metadata_{field}": remote_metadata.get(field) == expected
            for field, expected in metadata.items()
        },
    }
    if not all(checks.values()):
        raise IOError(
            f"기존 create-only 객체가 현재 결과와 다릅니다: {gs_uri(object_name)} / {checks}. "
            "기존 파일을 덮어쓰지 말고 OUTPUT_VERSION을 올리십시오."
        )
    PUBLISH_ACTIONS[object_name] = publish_action
    print(f"{publish_action}: {gs_uri(object_name)}")
    return {
        "object": object_name,
        "uri": gs_uri(object_name),
        "generation": int(blob.generation),
        "size": int(blob.size),
        "crc32c": blob.crc32c or "",
        "sha256": hashes["sha256"],
        "crc32c_local": hashes["crc32c"],
        "run_id": RUN_ID,
        "config_sha256": CONFIG_SHA256,
        "artifact_type": artifact_type,
        "parent_manifest_seal_sha256": PARENT_SEAL_SHA256,
        "parent_manifest_seal_generation": PARENT_SEAL_GENERATION,
        "row_count": None if row_count is None else int(row_count),
    }

def upload_json_create_only(payload, object_name, artifact_type):
    with tempfile.TemporaryDirectory() as temp_dir:
        local_path = Path(temp_dir) / "payload.json"
        write_json_file(local_path, payload)
        return upload_immutable(local_path, object_name, artifact_type)

artifact_specs = {
    "archive_counts": (
        LOCAL_TABLES / f"archive_counts_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/archive_counts_{OUTPUT_VERSION}.csv",
        "mission1_archive_counts", len(archive_counts),
    ),
    "type_range_counts": (
        LOCAL_TABLES / f"type_range_counts_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/type_range_counts_{OUTPUT_VERSION}.csv",
        "mission1_type_range_counts", len(type_range_counts),
    ),
    "type_counts": (
        LOCAL_TABLES / f"type_counts_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/type_counts_{OUTPUT_VERSION}.csv",
        "mission1_type_counts", len(type_counts),
    ),
    "obstruction_counts": (
        LOCAL_TABLES / f"obstruction_counts_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/obstruction_counts_{OUTPUT_VERSION}.csv",
        "mission1_obstruction_counts", len(obstruction_counts),
    ),
    "obstruction_plot_key": (
        LOCAL_TABLES / f"obstruction_plot_key_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/obstruction_plot_key_{OUTPUT_VERSION}.csv",
        "mission1_obstruction_plot_key", 15,
    ),
    "quality_summary": (
        LOCAL_TABLES / f"quality_summary_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/quality_summary_{OUTPUT_VERSION}.csv",
        "mission1_quality_summary", len(quality_summary),
    ),
    "split_summary": (
        LOCAL_TABLES / f"split_summary_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/split_summary_{OUTPUT_VERSION}.csv",
        "mission1_split_summary", len(split_summary),
    ),
    "split_stratum_balance": (
        LOCAL_TABLES / f"split_stratum_balance_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/split_stratum_balance_{OUTPUT_VERSION}.csv",
        "mission1_split_stratum_balance", len(stratum_balance),
    ),
    "bbox24_selection_csv": (
        LOCAL_TABLES / f"bbox24_selection_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/bbox24_selection_{OUTPUT_VERSION}.csv",
        "mission1_bbox24_selection_csv", len(selected),
    ),
    "bbox24_selection_parquet": (
        LOCAL_TABLES / f"bbox24_selection_{OUTPUT_VERSION}.parquet",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/bbox24_selection_{OUTPUT_VERSION}.parquet",
        "mission1_bbox24_selection_parquet", len(selected),
    ),
    "bbox24_boxes": (
        LOCAL_TABLES / f"bbox24_boxes_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/bbox24_boxes_{OUTPUT_VERSION}.csv",
        "mission1_bbox24_boxes", len(bbox_selected),
    ),
    "bbox24_range_io": (
        LOCAL_TABLES / f"bbox24_range_io_{OUTPUT_VERSION}.csv",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/tables/bbox24_range_io_{OUTPUT_VERSION}.csv",
        "mission1_bbox24_range_io", len(range_report),
    ),
    "type_counts_figure": (
        LOCAL_FIGURES / f"type_counts_{OUTPUT_VERSION}.png",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/figures/type_counts_{OUTPUT_VERSION}.png",
        "mission1_type_counts_figure", None,
    ),
    "obstruction_counts_figure": (
        LOCAL_FIGURES / f"obstruction_counts_{OUTPUT_VERSION}.png",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/figures/obstruction_counts_{OUTPUT_VERSION}.png",
        "mission1_obstruction_counts_figure", None,
    ),
    "bbox24_montage": (
        LOCAL_FIGURES / f"bbox24_montage_{OUTPUT_VERSION}.png",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/figures/bbox24_montage_{OUTPUT_VERSION}.png",
        "mission1_bbox24_montage", 24,
    ),
    "mission1_summary": (
        LOCAL_EDA / f"mission1_summary_{OUTPUT_VERSION}.json",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/mission1_summary_{OUTPUT_VERSION}.json",
        "mission1_summary", None,
    ),
    "storage_report": (
        LOCAL_EDA / f"storage_report_{OUTPUT_VERSION}.md",
        f"dcc/eda/mission1_{OUTPUT_VERSION}/storage_report_{OUTPUT_VERSION}.md",
        "mission1_storage_report", None,
    ),
    "splits": (
        SPLIT_LOCAL_PATHS["splits"],
        f"dcc/eval_private/manifests/splits_{OUTPUT_VERSION}.parquet",
        "private_splits", len(private_split),
    ),
    "train": (
        SPLIT_LOCAL_PATHS["train"],
        f"dcc/eval_private/manifests/train_{OUTPUT_VERSION}.parquet",
        "private_train", len(train_part),
    ),
    "val": (
        SPLIT_LOCAL_PATHS["val"],
        f"dcc/eval_private/manifests/val_{OUTPUT_VERSION}.parquet",
        "private_val", len(val_part),
    ),
    "test_truth": (
        SPLIT_LOCAL_PATHS["test_truth"],
        f"dcc/eval_private/manifests/test_truth_{OUTPUT_VERSION}.parquet",
        "private_test_truth", len(test_truth),
    ),
    "test_inference": (
        SPLIT_LOCAL_PATHS["test_inference"],
        f"dcc/inference/manifests/test_inference_{OUTPUT_VERSION}.parquet",
        "test_inference", len(test_inference),
    ),
}
split_first_names = ["splits", "train", "val", "test_truth", "test_inference"]
split_first_specs = {name: artifact_specs.pop(name) for name in split_first_names}
artifact_specs = {**split_first_specs, **artifact_specs}
for type_name, figure_path in zip(TYPE_ORDER, detail_figure_paths):
    artifact_specs[f"bbox_type_{type_name}"] = (
        figure_path,
        f"dcc/eda/mission1_{OUTPUT_VERSION}/figures/bbox_type_{type_name}_{OUTPUT_VERSION}.png",
        f"mission1_bbox_type_{type_name}", 3,
    )

ARTIFACT_DESCRIPTORS = {}
for name in split_first_names:
    local_path, object_name, artifact_type, row_count = artifact_specs[name]
    descriptor = upload_immutable(
        local_path, object_name, artifact_type, row_count=row_count
    )
    ARTIFACT_DESCRIPTORS[name] = descriptor

split_artifact_names = ["splits", "train", "val", "test_truth", "test_inference"]
split_seal_payload = {
    "status": MISSION1_STATUS,
    "schema_version": "dcc-split-schema-v2",
    "pipeline_version": PIPELINE_VERSION,
    "pipeline_build_sha256": PIPELINE_BUILD_SHA256,
    "run_id": RUN_ID,
    "output_version": OUTPUT_VERSION,
    "config_sha256": CONFIG_SHA256,
    "pipeline_contract": PIPELINE_CONTRACT,
    "parent_manifest_seal": {
        "object": PARENT_SEAL_OBJECT,
        "generation": PARENT_SEAL_GENERATION,
        "sha256": PARENT_SEAL_SHA256,
    },
    "seed": SPLIT_SEED,
    "target_counts": TARGET_COUNTS,
    "actual_counts": actual_counts,
    "raw_type_split_targets": RAW_TYPE_SPLIT_TARGETS,
    "raw_type_split_actual": {
        type_name: {
            split: int(actual_type_split_table.loc[type_name, split])
            for split in TARGET_COUNTS
        }
        for type_name in TYPE_ORDER
    },
    "type_target_policy": TYPE_TARGET_POLICY,
    "effective_trainable_counts": effective_counts,
    "group_policy": "exact image SHA-256",
    "group_leakage_max_split_count": group_split_max,
    "forced_train_group_count": len(forced_train_groups),
    "singleton_rebalance_moves": singleton_moves,
    "singleton_moves_by_type": singleton_moves_by_type,
    "split_assignment_sha256": split_assignment_sha256,
    "max_stratum_ratio_error": max_stratum_ratio_error,
    "artifacts": {
        name: ARTIFACT_DESCRIPTORS[name] for name in split_artifact_names
    },
    "assertions": {
        "raw_rows": len(private_split) == EXPECTED_RAW_ROWS,
        "unique_stems": private_split["stem"].is_unique,
        "counts_exact": actual_counts == TARGET_COUNTS,
        "type_split_counts_exact": type_split_counts_exact,
        "group_leakage_zero": group_split_max == 1,
        "forced_groups_train_only": split_df.loc[
            split_df["group_id"].isin(forced_train_groups), "split"
        ].eq("train").all(),
        "eligible_definition_exact": split_df["eligible_for_training"].eq(
            split_df["trainable"] & split_df["strict_decode_ok"]
        ).all(),
        "val_test_nontrainable_zero": (
            nontrainable_by_split["val"] == nontrainable_by_split["test"] == 0
        ),
        "val_test_strict_failures_zero": (
            strict_failures_by_split["val"] == strict_failures_by_split["test"] == 0
        ),
        "test_inference_truth_stems_exact": (
            test_inference["stem"].tolist() == test_truth["stem"].tolist()
        ),
        "test_inference_safe_schema": (
            list(test_inference.columns) == ["stem", "image_path", "type"]
        ),
    },
}
if not all(split_seal_payload["assertions"].values()):
    raise AssertionError("split seal assertion 실패")
split_seal_descriptor = upload_json_create_only(
    split_seal_payload,
    f"dcc/eval_private/manifests/split_seal_{OUTPUT_VERSION}.json",
    "split_seal",
)
ARTIFACT_DESCRIPTORS["split_seal"] = split_seal_descriptor
print("=== SPLIT GCS PUBLISH PASS ===")
print("Split Seal:", split_seal_descriptor["uri"])

for name, (local_path, object_name, artifact_type, row_count) in artifact_specs.items():
    if name in split_first_names:
        continue
    descriptor = upload_immutable(
        local_path, object_name, artifact_type, row_count=row_count
    )
    ARTIFACT_DESCRIPTORS[name] = descriptor

mission1_seal_payload = {
    "status": MISSION1_STATUS,
    "schema_version": "dcc-mission1-schema-v2",
    "pipeline_version": PIPELINE_VERSION,
    "pipeline_build_sha256": PIPELINE_BUILD_SHA256,
    "notebook_name": NOTEBOOK_NAME,
    "run_id": RUN_ID,
    "output_version": OUTPUT_VERSION,
    "project_id": PROJECT_ID,
    "bucket_name": BUCKET_NAME,
    "config_sha256": CONFIG_SHA256,
    "pipeline_contract": PIPELINE_CONTRACT,
    "parent_manifest_seal": {
        "object": PARENT_SEAL_OBJECT,
        "generation": PARENT_SEAL_GENERATION,
        "sha256": PARENT_SEAL_SHA256,
        "pair_index_sha256": validation["pair_index_sha256"],
        "source_inventory_sha256": PARENT_SEAL["source_inventory_sha256"],
    },
    "counts": {
        "raw_rows": len(core),
        "official_validation_rows": 0,
        "bbox_samples": len(selected),
        "split": actual_counts,
        "raw_type_split_targets": RAW_TYPE_SPLIT_TARGETS,
        "effective_trainable": effective_counts,
    },
    "split_assignment_sha256": split_assignment_sha256,
    "type_target_policy": TYPE_TARGET_POLICY,
    "artifacts": ARTIFACT_DESCRIPTORS,
    "assertions": mission1_summary["hard_assertions"],
    "original_inputs_modified": False,
}
mission1_seal_descriptor = upload_json_create_only(
    mission1_seal_payload,
    f"dcc/audits/mission1_seal_{OUTPUT_VERSION}.json",
    "mission1_seal",
)

def verify_published_descriptor(descriptor):
    blob = BUCKET.blob(
        descriptor["object"], generation=int(descriptor["generation"])
    )
    blob.reload()
    metadata = blob.metadata or {}
    checks = {
        "generation": int(blob.generation) == int(descriptor["generation"]),
        "size": int(blob.size) == int(descriptor["size"]),
        "crc32c": (blob.crc32c or "") == descriptor["crc32c"],
        "sha256_metadata": metadata.get("sha256") == descriptor["sha256"],
        "run_id_metadata": metadata.get("run_id") == RUN_ID,
        "config_metadata": metadata.get("config_sha256") == CONFIG_SHA256,
        "artifact_type_metadata": (
            metadata.get("artifact_type") == descriptor["artifact_type"]
        ),
    }
    if not all(checks.values()):
        raise IOError(f"GCS 게시 후 재검증 실패: {descriptor['uri']} / {checks}")

for name in sorted(ARTIFACT_DESCRIPTORS):
    verify_published_descriptor(ARTIFACT_DESCRIPTORS[name])
verify_published_descriptor(mission1_seal_descriptor)
gcs_console_url = (
    f"https://console.cloud.google.com/storage/browser/{BUCKET_NAME}/"
    f"dcc/eval_private/manifests?project={PROJECT_ID}"
)
print("=== MISSION 1 COMPLETE: GCS_PUBLISH_PASS ===")
print("Mission 1 final status:", MISSION1_STATUS)
print("Split:", actual_counts)
print("Type x split exact:", type_split_counts_exact)
for name in ["splits", "train", "val", "test_truth", "test_inference"]:
    print(f"GCS {name}: {ARTIFACT_DESCRIPTORS[name]['uri']}")
print("Split Seal:", split_seal_descriptor["uri"])
print("Mission 1 Seal:", mission1_seal_descriptor["uri"])
print("GCS Console:", gcs_console_url)
print("24 bbox samples: 8 types x 3")
print("/content는 임시 공간이며 위 GCS 객체가 영구 산출물입니다.")


CREATED: gs://YOUR_GCS_BUCKET/dcc/eval_private/manifests/splits_v3.parquet
CREATED: gs://YOUR_GCS_BUCKET/dcc/eval_private/manifests/train_v3.parquet
CREATED: gs://YOUR_GCS_BUCKET/dcc/eval_private/manifests/val_v3.parquet
CREATED: gs://YOUR_GCS_BUCKET/dcc/eval_private/manifests/test_truth_v3.parquet
CREATED: gs://YOUR_GCS_BUCKET/dcc/inference/manifests/test_inference_v3.parquet
CREATED: gs://YOUR_GCS_BUCKET/dcc/eval_private/manifests/split_seal_v3.json
=== SPLIT GCS PUBLISH PASS ===
Split Seal: gs://YOUR_GCS_BUCKET/dcc/eval_private/manifests/split_seal_v3.json
CREATED: gs://YOUR_GCS_BUCKET/dcc/eda/mission1_v3/tables/archive_counts_v3.csv
CREATED: gs://YOUR_GCS_BUCKET/dcc/eda/mission1_v3/tables/type_range_counts_v3.csv
CREATED: gs://YOUR_GCS_BUCKET/dcc/eda/mission1_v3/tables/type_counts_v3.csv
CREATED: gs://YOUR_GCS_BUCKET/dcc/eda/mission1_v3/tables/obstruction_counts_v3.csv
CREATED: gs://YOUR_GCS_BUCKET/dcc/eda/mission1_v3/tables/obstruction_plot_key_v3.csv
CREATED: gs://YOUR_GCS_BUCKET

## 완료 확인과 다음 단계

마지막 셀에서 아래 항목을 확인합니다.

- `=== MISSION 1 COMPLETE: GCS_PUBLISH_PASS ===`
- `Mission 1 final status: PASS` 또는 `PASS_WITH_QUARANTINE`
- split 수량: train 246,468 / val 30,808 / test 30,809
- A/B/C/E/F/G/H/S 각 유형의 `Type x split exact: True`
- 24 bbox samples: 8 types × 3
- `gs://버킷/dcc/eval_private/manifests/split_seal_v3.json` 생성
- `gs://버킷/dcc/audits/mission1_seal_v3.json` 생성

`train_v3.parquet`에는 전체 pair와 분할 계보를 보존하기 위해 격리 행도 들어 있습니다.
후속 모델 학습 입력은 반드시 `trainable & strict_decode_ok`를 적용하고, 탐지 학습은
추가로 `detector_trainable`을 적용하십시오.

상세 그림 8개를 사람이 확인해 빨간 bbox가 목자판 전체 영역을 감싸는지 검토하십시오.
이상이 발견되면 원본을 수정하지 말고 stem, annotation index, 판단을 별도 audit 로그에
남깁니다. 검토가 끝나면 같은 split Seal을 고정 입력으로 사용해 Mission 2 문제 정의와
Mission 3 baseline 학습 파일을 준비합니다.
